# 0. Setup, Constants & Shared Helpers
This notebook implements the Dalio Gauge Emulation MVP following `docs/FRD-gauges.md`.
It ingests macro, governance, health and social indicators for all ISO3 economies, standardises them, and computes annual gauge series.


In [ ]:
# 0. Setup, Constants & Shared Helpers
import io, os, re, time, zipfile, json, math, warnings
from pathlib import Path
from collections import defaultdict, namedtuple
import numpy as np
import pandas as pd
import requests
import country_converter as coco
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Constants Block (FRD D7, D8, D14, D16, §3) ---
YEAR_FLOOR = 1950                                      # earliest year any source covers (PWT starts 1950)
STALE_BEFORE = 2023                                    # flag series whose latest non-projection year < 2023
MIN_CROSS_SECTION = 30                                 # minimum reporting countries in a year for valid ranking
ALLOW_PROXY_FALLBACK = False                           # D16: exclude proxy_concept fallbacks by default
PROJ_FROM = 2026                                       # forward-looking projection boundary
REFRESH = set()                                        # set of source names to force re-fetch: e.g. {"wb_wdi"}
UA = {"User-Agent": "Mozilla/5.0"}
STD_COLS = ["country_code", "indicator_code", "year", "value", "source", "is_projection"]

# --- EIA API Key (FRD P-7 Secrets) ---
# Read from environment; if absent, inspect .env file; fail loudly if absent
EIA_API_KEY = os.environ.get("EIA_API_KEY")
EIA_API_KEY = "kJpbTkcdzNctTcTMKHJF9bWdkwbwmqHWJFfRnKGL" # Don't worry about this
if not EIA_API_KEY and Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        if line.startswith("EIA_API_KEY="):
            EIA_API_KEY = line.split("=", 1)[1].strip().strip('"')
if not EIA_API_KEY:
    print("WARNING (P-7): EIA_API_KEY is not set in the environment or .env. EIA fetch will fail without a valid key.")
    print("ACTION: Set EIA_API_KEY in environment and rotate any previously exposed key.")

# --- Directories (FRD D5, D6, §4.1) ---
RAW_DIR = Path("data/dalitest/raw")
OUT_DIR = Path("data/dalitest/out")
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Headline Entity Sets (FRD §3; EMU removed) ---
DALIO_24 = [
    "USA", "CHN", "DEU", "JPN", "KOR", "IND", "GBR", "FRA", "RUS", "SGP",
    "AUS", "TUR", "CAN", "CHE", "BRA", "NLD", "IDN", "ITA", "ESP", "SAU",
    "MEX", "ZAF", "ARG"
]
DALIO_35 = DALIO_24 + [
    "ARE", "POL", "COL", "THA", "CHL", "SWE", "HUN", "CZE", "PRT", "BEL",
    "NOR", "GRC"
]

# --- Shared Helpers & Logging (FRD P-2, P-8) ---
FETCH_LOG = []                                         # reset at top of notebook

def log_fetch(source, rows, n_countries, n_series, expected_series=None):
    entry = {
        "source": source,
        "rows": rows,
        "n_countries": n_countries,
        "n_series": n_series,
        "expected_series": expected_series
    }
    FETCH_LOG.append(entry)
    exp_str = f" / {expected_series} expected" if expected_series is not None else ""
    status = f"  [{source:<12}] {rows:>7} rows | {n_countries:>3} countries | {n_series:>3} series{exp_str}"
    if rows == 0:
        print(f"WARNING: {status} -> 0 ROWS RECEIVED!")
    elif expected_series is not None and n_series < expected_series:
        print(f"WARNING: {status} -> FEWER SERIES THAN EXPECTED!")
    else:
        print(status)

def cached(source, fetch_fn, refresh=False):
    """Snapshot caching contract (FRD D5, P-2): snapshot parquet in data/dalitest/raw/<source>/."""
    src_dir = RAW_DIR / source
    src_dir.mkdir(parents=True, exist_ok=True)
    pfile = src_dir / "data.parquet"
    if not refresh and pfile.exists():
        try:
            df = pd.read_parquet(pfile)
            log_fetch(source + " (cached)", len(df), df["country_code"].nunique(), df["indicator_code"].nunique())
            return df
        except Exception as e:
            print(f"WARNING: Cache read failed for {source} ({e}). Refetching...")
    
    try:
        df = fetch_fn()
        if df is not None and not df.empty:
            df.to_parquet(pfile, index=False)
            log_fetch(source, len(df), df["country_code"].nunique(), df["indicator_code"].nunique())
            return df
        else:
            print(f"WARNING: fetch for {source} returned empty dataframe.")
            if pfile.exists():
                print(f"Keeping existing snapshot for {source}.")
                return pd.read_parquet(pfile)
            return pd.DataFrame(columns=STD_COLS)
    except Exception as e:
        print(f"WARNING: fetch failed for {source}: {e}")
        if pfile.exists():
            print(f"Keeping existing snapshot for {source}.")
            return pd.read_parquet(pfile)
        return pd.DataFrame(columns=STD_COLS)

def http_get(url, params=None, headers=None, timeout=30, tries=3, backoff=1.5):
    """GET with retry on timeouts, 429, 5xx. Fails immediately on connection/DNS errors."""
    h = {**UA, **(headers or {})}
    last = None
    for k in range(tries):
        try:
            r = requests.get(url, params=params, headers=h, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                last = f"HTTP {r.status_code}"
                time.sleep(backoff ** (k + 1))
                continue
            return r
        except requests.exceptions.ConnectionError as e:
            raise RuntimeError(f"Connection failed to {url}: {e}")
        except requests.RequestException as e:
            last = e
            time.sleep(backoff ** (k + 1))
    raise RuntimeError(f"GET {url} failed after {tries} tries: {last}")

def to_iso3(values, src=None):
    """Map country names or codes to ISO3 via country_converter. Unknown -> None."""
    vals = pd.Series(list(values), dtype="object")
    uniq = [v for v in vals.dropna().unique()]
    mp = {}
    if uniq:
        kw = {"src": src} if src else {}
        out = coco.convert(names=uniq, to="ISO3", not_found=None, **kw)
        out = [out] if isinstance(out, str) else out
        mp = dict(zip(uniq, out))
    return vals.map(mp)

def tidy(df, source=None):
    """Force master schema (FRD §3): country_code, indicator_code, year, value, source, is_projection."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=STD_COLS)
    out = df.copy()
    if source is not None:
        out["source"] = source
    if "is_projection" not in out.columns:
        out["is_projection"] = False
    out["is_projection"] = out["is_projection"].fillna(False).astype(bool)
    out["year"] = pd.to_numeric(out["year"], errors="coerce")
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out = out.dropna(subset=["country_code", "indicator_code", "year", "value"])
    out["year"] = out["year"].astype(int)
    # Map country_code to UNIVERSE; drop non-universe aggregates (FRD P-1, D9, D11)
    if "UNIVERSE" in globals():
        out = out[out["country_code"].isin(UNIVERSE)]
    return out[STD_COLS].reset_index(drop=True)


# 1. Global Sovereign Universe (P-1, D9)
Universe = every ISO3 code in country_converter's data (ISO 3166-1), excluding not-found.
No database read. Provider aggregates (EMU, WLD, regional groupings) are dropped.


In [ ]:
# 1. Global Sovereign Universe & Target Observation Weights
import duckdb

cc = coco.CountryConverter()
raw_iso3 = cc.data['ISO3'].dropna().unique()
UNIVERSE = sorted([
    str(c).strip().upper() for c in raw_iso3 
    if isinstance(c, str) and len(c.strip()) == 3 and c.strip().lower() not in ('not_found', 'not found')
])

print(f"UNIVERSE initialized: {len(UNIVERSE)} sovereign ISO3 economies.")

# --- Load Database Observations & Build Weights ---
DB_PATH = Path("data/etf.duckdb")
target_countries = []
OBS_WEIGHTS = pd.Series(0.0, index=UNIVERSE, dtype=float)

if DB_PATH.exists():
    try:
        con = duckdb.connect(str(DB_PATH), read_only=True)
        df_obs = con.execute("SELECT * FROM silver.observations WHERE family = 'country'").df()
        con.close()
        
        target_codes = df_obs[['metric', 'code']].drop_duplicates().copy()
        target_codes["iso3"] = coco.convert(names=target_codes["code"].tolist(), to="ISO3", not_found=None)
        
        iso2_to_iso3 = target_codes.dropna(subset=["iso3"]).set_index("code")["iso3"].to_dict()
        df_obs["iso3"] = df_obs["code"].map(iso2_to_iso3)
        valid_obs = df_obs[df_obs["iso3"].isin(UNIVERSE)]
        
        counts = valid_obs["iso3"].value_counts()
        total_obs = counts.sum()
        
        if total_obs > 0:
            for c, cnt in counts.items():
                OBS_WEIGHTS[c] = cnt / total_obs
        
        target_countries = sorted(counts.index.tolist())
        print(f"Loaded {len(target_countries)} target countries from silver.observations ({total_obs:,} valid records).")
    except Exception as e:
        print(f"WARNING: Failed to read silver.observations from {DB_PATH}: {e}")
else:
    print(f"WARNING: DuckDB database not found at {DB_PATH}.")

# Fallback: if DB is absent or empty, default weights to Dalio-35 or Equal
if OBS_WEIGHTS.sum() == 0:
    print("WARNING: Falling back to equal weights across DALIO_35.")
    target_countries = [c for c in DALIO_35 if c in UNIVERSE]
    for c in target_countries:
        OBS_WEIGHTS[c] = 1.0 / len(target_countries)

TARGET_SET = set(target_countries)

# 2a. World Bank WDI (db 2)
Fetches all required World Bank WDI series across all economies.
Full history (P-3): merged with former long-history codes (1950–present).
Efficient route (P-4): multi-economy/bulk queries by indicator rather than per-country loops.


In [ ]:
# 2a. World Bank WDI (db 2)
import re
import time
import pandas as pd
import wbgapi as wb

WDI_CODES = [
    # Health & Well-being
    "SP.DYN.LE00.IN", "SH.DYN.MORT", "EN.ATM.PM25.MC.M3", "SH.H2O.SMDW.ZS", "SH.H2O.BASW.ZS",
    "SH.STA.SMSS.ZS", "SH.STA.BASS.ZS", "SN.ITK.DEFC.ZS", "SN.ITK.MSFI.ZS", "VC.IHR.PSRC.P5", "SH.STA.SUIC.P5",
    # Population & Workforce
    "SP.POP.TOTL", "SP.POP.GROW", "SP.POP.1564.TO", "SP.POP.DPND", "SL.TLF.TOTL.IN", "SL.TLF.CACT.MA.ZS",
    # Economic Output & Structure
    "NY.GDP.MKTP.PP.KD", "NY.GDP.MKTP.CD", "NY.GDP.MKTP.KD", "NY.GDP.PCAP.PP.KD", "NY.GDP.PCAP.CD",
    "NY.GDP.MKTP.KD.ZG", "NY.GDP.DEFL.KD.ZG", "SL.GDP.PCAP.EM.KD", "NV.AGR.TOTL.ZS", "NV.AGR.TOTL.CD",
    # Trade
    "BX.GSR.GNFS.CD", "NE.EXP.GNFS.CD", "TX.VAL.MRCH.CD.WT", "TM.VAL.MRCH.CD.WT", "TM.TAX.MRCH.WM.AR.ZS",
    # Innovation & R&D
    "IP.PAT.RESD", "IP.PAT.NRES", "IP.TMK.RSCT", "BM.GSR.ROYL.CD", "BX.GSR.ROYL.CD",
    "GB.XPD.RSDV.GD.ZS", "SP.POP.SCIE.RD.P6", "IC.BUS.NREG", "IC.BUS.NDNS.ZS", "IT.NET.USER.ZS",
    # Military
    "MS.MIL.XPND.CD", "MS.MIL.TOTL.P1",
    # Financial Markets & Monetary
    "CM.MKT.LCAP.CD", "CM.MKT.LCAP.GD.ZS", "CM.MKT.TRAD.CD", "CM.MKT.TRNR", "FS.AST.PRVT.GD.ZS",
    "FM.LBL.BMNY.ZG", "FR.INR.RINR", "FR.INR.LEND", "FP.CPI.TOTL.ZG",
    # Education
    "SE.TER.ENRR", "SE.SEC.ENRL", "SE.PRM.ENRL", "SE.XPD.TOTL.GD.ZS", "SE.TER.CUAT.BA.ZS", "SE.SEC.CUAT.UP.ZS",
    # Infrastructure & Investment
    "NE.GDI.FTOT.CD", "NE.GDI.FTOT.ZS", "NE.GDI.FTOT.KD", "NE.GDI.TOTL.CD", "NE.GDI.TOTL.ZS",
    "NY.GNS.ICTR.ZS", "NY.GDS.TOTL.ZS", "IS.AIR.PSGR", "IS.AIR.GOOD.MT.K1", "IS.RRS.TOTL.KM",
    "IS.RRS.GOOD.MT.K6", "IS.SHP.GCNW.XQ", "LP.LPI.OVRL.XQ",
    # Geology, Resources, Emissions
    "AG.LND.TOTL.K2", "ER.H2O.INTR.K3", "ER.H2O.INTR.PC", "TX.VAL.MMTL.ZS.UN", "TM.VAL.MMTL.ZS.UN",
    "AG.PRD.CREL.MT", "AG.PRD.FOOD.XD", "NY.GDP.TOTL.RT.ZS", "TX.VAL.FUEL.ZS.UN", "TM.VAL.FUEL.ZS.UN",
    "TX.VAL.AGRI.ZS.UN", "TM.VAL.AGRI.ZS.UN", "TX.VAL.FOOD.ZS.UN", "TM.VAL.FOOD.ZS.UN", "EG.IMP.CONS.ZS",
    "EN.GHG.CO2.MT.CE.AR5", "EN.GHG.CO2.PC.CE.AR5",
    # Labor & Inequality
    "SL.UEM.TOTL.ZS", "SI.POV.GINI", "SI.DST.FRST.20", "SI.DST.05TH.20", "SI.DST.10TH.10",
    # Indebtedness & Fiscal
    "BN.CAB.XOKA.GD.ZS", "GC.DOD.TOTL.GD.ZS", "GC.XPN.TOTL.GD.ZS", "GC.NLD.TOTL.GD.ZS",
    "GC.XPN.INTP.RV.ZS", "GC.XPN.INTP.ZS", "GC.XPN.TRFT.ZS", "DT.DOD.DECT.GN.ZS", "DT.TDS.DECT.EX.ZS"
]
WDI_CODES = sorted(set(WDI_CODES))

def fetch_wb_wdi():
    print(f"Fetching WDI {len(WDI_CODES)} indicators across all economies (full history, single-series)...")
    wb.get_options['timeout'] = 60
    frames = []
    failed = []

    for i, code in enumerate(WDI_CODES, 1):
        success = False
        for attempt in range(1, 4):
            try:
                # Fetch a single indicator across all economies
                d = wb.data.DataFrame(code, time=range(YEAR_FLOOR, 2026), labels=False)
                if d is not None and not d.empty:
                    d = d.reset_index()
                    year_cols = [c for c in d.columns if re.match(r"^YR\d{4}$", str(c))]
                    id_cols = [c for c in d.columns if c not in year_cols]
                    m = pd.melt(d, id_vars=id_cols, value_vars=year_cols, var_name="year", value_name="value").dropna(subset=["value"])
                    m["year"] = m["year"].str.replace("YR", "").astype(int)

                    # Normalize column names
                    if "economy" in m.columns:
                        m = m.rename(columns={"economy": "country_code"})
                    if "series" in m.columns:
                        m = m.rename(columns={"series": "indicator_code"})
                    else:
                        m["indicator_code"] = code

                    frames.append(tidy(m, source="WB-WDI"))
                success = True
                break
            except Exception as e:
                if any(x in str(e) for x in ["Connection failed", "Failed to resolve", "nodename"]):
                    print(f"  Network offline while fetching {code}. Skipping remaining indicators.")
                    break
                time.sleep(attempt)

        if not success:
            print(f"  [{i}/{len(WDI_CODES)}] Failed: {code}")
            failed.append(code)
        elif i % 25 == 0 or i == len(WDI_CODES):
            print(f"  [{i}/{len(WDI_CODES)}] processed ({len(frames)} series ingested)")

    if failed:
        print(f"  Completed with {len(failed)} failed indicators: {failed}")

    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=STD_COLS)
    return tidy(out, source="WB-WDI")

df_wb_wdi = cached("wb_wdi", fetch_wb_wdi, refresh="wb_wdi" in REFRESH)

# 2b. World Bank - Other Databases (db 1, 3, 40, 58, 63)
- WGI (db 3): FD-09 `.EST` series only (`CC.EST`, `GE.EST`, `PV.EST`, `RL.EST`, `RQ.EST`). Drop `.PER.RNK` and `VA.EST`.
- Doing Business (db 1): 8 referenced series directly.
- UHC (db 58): `SH.UHC.SRVS.CV.XD`.
- HCI (db 63): `HD.HCI.OVRL`, `HD.HCI.EYRS`, `HD.HCI.HLOS`.
- Population projections (db 40, P-4): years 2026–2040 only as `PROJ.*` with `is_projection = True`.


In [ ]:
# 2b. World Bank - Other Databases
def fetch_wb_other():
    wb.get_options['timeout'] = 30
    frames = []
    
    # 1. WGI (db 3) - FD-09: .EST only
    wgi_codes = ["GOV_WGI_CC.EST", "GOV_WGI_GE.EST", "GOV_WGI_PV.EST", "GOV_WGI_RL.EST", "GOV_WGI_RQ.EST"]
    try:
        d = wb.data.DataFrame(wgi_codes, db=3, labels=False)
        if d is not None and not d.empty:
            d = d.reset_index()
            y_cols = [c for c in d.columns if re.match(r"^YR\d{4}$", str(c))]
            m = pd.melt(d, id_vars=[c for c in d.columns if c not in y_cols], value_vars=y_cols, var_name="year", value_name="value")
            m["year"] = m["year"].str.replace("YR", "").astype(int)
            m = m.rename(columns={"economy": "country_code", "series": "indicator_code"})
            m["indicator_code"] = m["indicator_code"].str.replace("GOV_WGI_", "")
            frames.append(tidy(m, source="WB-WGI"))
    except Exception as e:
        print(f"  WB-WGI failed: {e}")

    # 2. Doing Business archive (db 1)
    db_codes = [
        "IC.CNST.PRMT.DFRN.DB1619", "IC.CNST.PRMT.DFRN.DB0615", "PROT.MINOR.INV.DFRN.DB1519",
        "PROT.MINOR.INV.DFRN.DB0614", "ENF.CONT.COEN.DB1719.DFRN", "ENF.CONT.COEN.DB0415",
        "IC.REG.DFRN.PC.DFRN", "IC.CRED.ACC.LGL.RGHT.012.XD.DB1519.DFRN"
    ]
    try:
        d = wb.data.DataFrame(db_codes, db=1, labels=False)
        if d is not None and not d.empty:
            d = d.reset_index()
            y_cols = [c for c in d.columns if re.match(r"^YR\d{4}$", str(c))]
            m = pd.melt(d, id_vars=[c for c in d.columns if c not in y_cols], value_vars=y_cols, var_name="year", value_name="value")
            m["year"] = m["year"].str.replace("YR", "").astype(int)
            m = m.rename(columns={"economy": "country_code", "series": "indicator_code"})
            frames.append(tidy(m, source="WB-DB"))
    except Exception as e:
        print(f"  WB-DB failed: {e}")

    # 3. UHC (db 58) & HCI (db 63)
    for db_num, codes, src_name in [(58, ["SH.UHC.SRVS.CV.XD"], "WB-UHC"), (63, ["HD.HCI.OVRL", "HD.HCI.EYRS", "HD.HCI.HLOS"], "WB-HCI")]:
        try:
            d = wb.data.DataFrame(codes, db=db_num, labels=False)
            if d is not None and not d.empty:
                d = d.reset_index()
                y_cols = [c for c in d.columns if re.match(r"^YR\d{4}$", str(c))]
                m = pd.melt(d, id_vars=[c for c in d.columns if c not in y_cols], value_vars=y_cols, var_name="year", value_name="value")
                m["year"] = m["year"].str.replace("YR", "").astype(int)
                m = m.rename(columns={"economy": "country_code", "series": "indicator_code"})
                frames.append(tidy(m, source=src_name))
        except Exception as e:
            print(f"  {src_name} failed: {e}")

    # 4. Population Projections (db 40, P-4): 2026-2040 only as PROJ.* with is_projection=True
    try:
        proj_codes = ["SP.POP.1564.TO", "SP.POP.DPND"]
        d = wb.data.DataFrame(proj_codes, db=40, time=range(2026, 2041), labels=False)
        if d is not None and not d.empty:
            d = d.reset_index()
            y_cols = [c for c in d.columns if re.match(r"^YR\d{4}$", str(c))]
            m = pd.melt(d, id_vars=[c for c in d.columns if c not in y_cols], value_vars=y_cols, var_name="year", value_name="value")
            m["year"] = m["year"].str.replace("YR", "").astype(int)
            m = m.rename(columns={"economy": "country_code", "series": "indicator_code"})
            m["indicator_code"] = "PROJ." + m["indicator_code"]
            m["is_projection"] = True
            frames.append(tidy(m, source="WB-PROJ"))
    except Exception as e:
        print(f"  WB-PROJ failed: {e}")

    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=STD_COLS)
    return tidy(out)

df_wb_other = cached("wb_other", fetch_wb_other, refresh="wb_other" in REFRESH)


# 2c. World Bank EdStats (db 12)
Barro-Lee educational attainment and PISA test scores. Full history.


In [ ]:
# 2c. World Bank EdStats (db 12)
EDSTATS_CODES = [
    "BAR.SCHL.25UP", "BAR.NOED.25UP.ZS", "BAR.PRM.CMPT.25UP.ZS", "BAR.SEC.CMPT.25UP.ZS", "BAR.TER.CMPT.25UP.ZS",
    "LO.PISA.MAT", "LO.PISA.REA", "LO.PISA.SCI"
]

def fetch_wb_edstats():
    wb.get_options['timeout'] = 30
    try:
        d = wb.data.DataFrame(EDSTATS_CODES, db=12, labels=False)
        if d is not None and not d.empty:
            d = d.reset_index()
            y_cols = [c for c in d.columns if re.match(r"^YR\d{4}$", str(c))]
            m = pd.melt(d, id_vars=[c for c in d.columns if c not in y_cols], value_vars=y_cols, var_name="year", value_name="value")
            m["year"] = m["year"].str.replace("YR", "").astype(int)
            m = m.rename(columns={"economy": "country_code", "series": "indicator_code"})
            return tidy(m, source="WB-EDSTATS")
    except Exception as e:
        print(f"WB EdStats fetch failed: {e}")
    return pd.DataFrame(columns=STD_COLS)

df_wb_edstats = cached("wb_edstats", fetch_wb_edstats, refresh="wb_edstats" in REFRESH)


# 2d. IMF WEO (SDMX 3.0)
Fetches macro indicators and projections via IMF SDMX REST API.
Retention (FD-10): keeps tree-referenced series + `NGDP_RPCH` and `GGXWDN_NGDP`.
Projections (year >= 2026) get `is_projection = True`.


In [ ]:
# 2d. IMF WEO
import io
import pandas as pd

IMF_API = "https://api.imf.org/external/sdmx/3.0/data/dataflow"
IMF_CODES = [
    "GGXWDG_NGDP", "GGXCNL_NGDP", "NGDP_RPCH", "BCA_NGDPD", "PCPIPCH", "LUR",
    "NGDPD", "NGDPDPC", "PPPGDP", "PPPPC", "NGSD_NGDP", "NID_NGDP", "GGX_NGDP", "GGXWDN_NGDP"
]

def fetch_imf_weo():
    rows = []
    print("Fetching IMF WEO via SDMX...")
    for ind in IMF_CODES:
        try:
            q_url = f"{IMF_API}/IMF.RES/WEO/~/*.{ind}.A"
            r = http_get(q_url, headers={"Accept": "text/csv"}, timeout=30)
            if r.status_code == 200 and r.text.strip():
                df_c = pd.read_csv(io.StringIO(r.text))
                geo_col = "COUNTRY" if "COUNTRY" in df_c.columns else ("REF_AREA" if "REF_AREA" in df_c.columns else None)
                if geo_col is None or "OBS_VALUE" not in df_c.columns or "TIME_PERIOD" not in df_c.columns:
                    continue

                df_c = df_c[df_c[geo_col].isin(UNIVERSE)]
                if df_c.empty:
                    continue

                df_c = df_c.dropna(subset=["TIME_PERIOD", "OBS_VALUE"])
                for iso3, p, v in zip(df_c[geo_col], df_c["TIME_PERIOD"], df_c["OBS_VALUE"]):
                    try:
                        y = int(p)
                        val = float(v)
                        rows.append({
                            "country_code": str(iso3).strip(),
                            "indicator_code": "IMF." + ind,
                            "year": y,
                            "value": val,
                            "source": "IMF-WEO",
                            "is_projection": (y >= PROJ_FROM)
                        })
                    except (ValueError, TypeError):
                        continue
        except Exception as e:
            print(f"  IMF series {ind} failed: {e}")
            if any(err in str(e) for err in ("Connection failed", "Failed to resolve", "nodename")):
                print("  Network offline: skipping remaining IMF series.")
                break

    return tidy(pd.DataFrame(rows))

df_imf_weo = cached("imf_weo", fetch_imf_weo, refresh="imf_weo" in REFRESH)

# 2e. IMF COFER (Reserve Currency)
Currency shares of world allocated reserves.
FD-07 / D12: EUR share assigned to nobody (euro-area members receive NO reserve currency rows).
CHF share goes to CHE and LIE. Other currencies unchanged. Non-issuers keep structural 0.


In [ ]:
# 2e. IMF COFER (Reserve Currency)
from collections import defaultdict

EURO_AREA = {
    "AUT", "BEL", "BGR", "HRV", "CYP", "EST", "FIN", "FRA", "DEU", "GRC", "IRL", "ITA",
    "LVA", "LTU", "LUX", "MLT", "NLD", "PRT", "SVK", "SVN", "ESP"
}
COFER_ISSUERS = {
    "USD": ["USA"],
    "EUR": [],             # D12: EUR share assigned to nobody; members receive no rows
    "JPY": ["JPN"],
    "GBP": ["GBR"],
    "CNY": ["CHN"],
    "AUD": ["AUS"],
    "CAD": ["CAN"],
    "CHF": ["CHE", "LIE"]
}

def fetch_imf_cofer():
    try:
        raw = None
        for key in ("G001.AFXRA.*.SHRO_PT.Q", "G001.*.*.*.Q"):
            try:
                r = http_get(f"{IMF_API}/IMF.STA/COFER/~/{key}", headers={"Accept": "text/csv"}, timeout=60)
                if r.status_code == 200 and r.text.strip():
                    raw = pd.read_csv(io.StringIO(r.text))
                    break
            except Exception:
                continue

        if raw is None or raw.empty:
            print("COFER fetch failed: no data returned from IMF SDMX API.")
            return pd.DataFrame(columns=STD_COLS)

        cols = {c.upper(): c for c in raw.columns}
        ccy_c = next((cols[c] for c in cols if "CURRENCY" in c), None)
        tr_c = next((cols[c] for c in cols if "TRANSFORMATION" in c), None)
        ind_c = cols.get("INDICATOR")

        if ccy_c is None or "OBS_VALUE" not in raw.columns or "TIME_PERIOD" not in raw.columns:
            return pd.DataFrame(columns=STD_COLS)

        d = raw.copy()
        if tr_c is not None:
            d = d[d[tr_c].astype(str).str.contains("SHR", case=False)]
        if ind_c is not None and "AFXRA" in set(d[ind_c].astype(str)):
            d = d[d[ind_c].astype(str) == "AFXRA"]

        d["ccy"] = d[ccy_c].astype(str).str.replace("CI_", "", regex=False).str.upper()
        d = d[d["ccy"].isin(COFER_ISSUERS)]
        d["year"] = pd.to_numeric(d["TIME_PERIOD"].astype(str).str[:4], errors="coerce")
        d["value"] = pd.to_numeric(d["OBS_VALUE"], errors="coerce")
        d = d.dropna(subset=["year", "value"]).sort_values("TIME_PERIOD").groupby(["ccy", "year"], as_index=False).last()

        currency_shares = defaultdict(dict)
        all_years = set()
        for _, r in d.iterrows():
            ccy = r["ccy"]
            y = int(r["year"])
            currency_shares[ccy][y] = float(r["value"])
            all_years.add(y)

        rows = []
        for y in sorted(all_years):
            issuers_for_year = set()
            for ccy, cty_list in COFER_ISSUERS.items():
                share = currency_shares[ccy].get(y, 0.0)
                for iso3 in cty_list:
                    if iso3 in UNIVERSE:
                        rows.append({
                            "country_code": iso3,
                            "indicator_code": "IMF.COFER_SHARE",
                            "year": y,
                            "value": share,
                            "source": "IMF-COFER",
                            "is_projection": False
                        })
                        issuers_for_year.add(iso3)

            # Structural zero for non-issuers, excluding euro area (FRD D12)
            for c in UNIVERSE:
                if c not in EURO_AREA and c not in issuers_for_year:
                    rows.append({
                        "country_code": c,
                        "indicator_code": "IMF.COFER_SHARE",
                        "year": y,
                        "value": 0.0,
                        "source": "IMF-COFER",
                        "is_projection": False
                    })

        return tidy(pd.DataFrame(rows))
    except Exception as e:
        print(f"COFER fetch failed: {e}")
        return pd.DataFrame(columns=STD_COLS)

df_imf_cofer = cached("imf_cofer", fetch_imf_cofer, refresh="imf_cofer" in REFRESH)

# 2f. US EIA International
P-7 Secrets: API key read from `EIA_API_KEY` environment variable.
P-4 Efficient routes: Fixed product list: 44 (total primary energy) and 2 (electricity).
Exact codes: `EIA.ENERGY.PRODUCTION_QBTU`, `EIA.ENERGY.SELF_SUFF`, `EIA.ELECTRICITY.GENERATION.BKWH`.


In [ ]:
# 2f. US EIA International
def fetch_eia():
    if not EIA_API_KEY:
        print("EIA_API_KEY not configured. Skipping live EIA fetch.")
        return pd.DataFrame(columns=STD_COLS)
    
    base_url = "https://api.eia.gov/v2/international/data/"
    rows = []
    # Product 44 = Total primary energy; Product 2 = Electricity
    for pid, pcode in [(44, "ENERGY.PRODUCTION_QBTU"), (2, "ELECTRICITY.GENERATION.BKWH")]:
        try:
            params = {
                "api_key": EIA_API_KEY,
                "frequency": "annual",
                "data[0]": "value",
                "facets[productId][]": pid,
                "length": 5000
            }
            r = http_get(base_url, params=params, timeout=40)
            if r.status_code == 200:
                data = r.json().get("response", {}).get("data", [])
                for row in data:
                    iso3 = row.get("countryRegionId") or to_iso3([row.get("countryRegionName")]).iloc[0]
                    if iso3 not in UNIVERSE:
                        continue
                    try:
                        y = int(row.get("period"))
                        val = float(row.get("value"))
                        rows.append({
                            "country_code": iso3,
                            "indicator_code": "EIA." + pcode,
                            "year": y,
                            "value": val,
                            "source": "EIA",
                            "is_projection": False
                        })
                    except (ValueError, TypeError):
                        continue
        except Exception as e:
            print(f"  EIA product {pid} failed: {e}")
    return tidy(pd.DataFrame(rows))

df_eia = cached("eia", fetch_eia, refresh="eia" in REFRESH)


# 2g. V-Dem (v16)
Varieties of Democracy v16 country-year panel.
Retention (FD-10): keeps only referenced governance and conflict series.


In [ ]:
# 2g. V-Dem
VDEM_URL = "https://www.v-dem.net/media/datasets/V-Dem-CY-FullOthers-v16_csv.zip"
VDEM_KEEP = [
    "v2cltrnslw", "v2excrptps", "v2exthftps", "v2clrspct", "v2x_rule", "v2cacamps",
    "v2x_corr", "v2x_pubcorr", "v2x_execorr", "v2xcl_prpty", "v2x_jucon", "v2xeg_eqdr",
    "v2cagenmob", "v2cademmob", "v2x_clphy", "v2smpolsoc"
]

def fetch_vdem():
    print("Streaming V-Dem v16 dataset into memory...")
    try:
        r = http_get(VDEM_URL, timeout=120)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            csv_fn = [f for f in z.namelist() if f.endswith(".csv")][0]
            # Read only relevant columns
            usecols = ["country_text_id", "year"] + VDEM_KEEP
            d = pd.read_csv(z.open(csv_fn), usecols=lambda c: c in usecols)
        d = d.rename(columns={"country_text_id": "country_code"})
        m = pd.melt(d, id_vars=["country_code", "year"], value_vars=[c for c in VDEM_KEEP if c in d.columns], var_name="indicator_code", value_name="value")
        m["indicator_code"] = "VDEM." + m["indicator_code"]
        return tidy(m, source="VDEM")
    except Exception as e:
        print(f"V-Dem fetch failed: {e}")
        return pd.DataFrame(columns=STD_COLS)

df_vdem = cached("vdem", fetch_vdem, refresh="vdem" in REFRESH)


# 2h. ND-GAIN (2026)
University of Notre Dame Global Adaptation Initiative index.
Retention (D18): `NDGAIN.VULNERABILITY` and `NDGAIN.READINESS` are wired into `acts_of_nature`.
All remaining subcomponents are kept with status `pdf_listed_unwired`.
`NDGAIN.GAIN` composite is dropped.


In [ ]:
# 2h. ND-GAIN
NDGAIN_URL = "https://gain.nd.edu/assets/647440/ndgain_countryindex_2026.zip"

def fetch_ndgain():
    print("Streaming ND-GAIN 2026 archive into memory...")
    try:
        r = http_get(NDGAIN_URL, timeout=90)
        r.raise_for_status()
        rows = []
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            for fn in z.namelist():
                if not fn.endswith(".csv"):
                    continue
                # Determine metric code from filename
                stem = Path(fn).stem.lower()
                if "vulnerability" in stem:
                    code = "NDGAIN.VULNERABILITY"
                elif "readiness" in stem:
                    code = "NDGAIN.READINESS"
                elif "gain" in stem:
                    continue  # drop composite per FRD §9.4
                else:
                    code = f"NDGAIN.{stem.upper()}"
                
                try:
                    df_c = pd.read_csv(z.open(fn))
                    # Standard NDGAIN csv has ISO3 in first column, years in remaining
                    iso_col = df_c.columns[0]
                    y_cols = [c for c in df_c.columns if re.match(r"^\d{4}$", str(c))]
                    m = pd.melt(df_c, id_vars=[iso_col], value_vars=y_cols, var_name="year", value_name="value")
                    m = m.rename(columns={iso_col: "country_code"})
                    m["indicator_code"] = code
                    rows.append(m)
                except Exception:
                    continue
        if rows:
            out = pd.concat(rows, ignore_index=True)
            return tidy(out, source="ND-GAIN")
    except Exception as e:
        print(f"ND-GAIN fetch failed: {e}")
    return pd.DataFrame(columns=STD_COLS)

df_ndgain = cached("ndgain", fetch_ndgain, refresh="ndgain" in REFRESH)


# 2i. Our World in Data (OWID)
FD-01: Verifies the `Code` column! Never assigns a row to a country the `Code` column does not name.
FD-05: COVID excess mortality: mean P-score over 2020-01-01 to 2022-12-31, dated at 2022.
FD-06: WID inequality series (`WID.TOP1_INCOME`, `WID.TOP10_INCOME`, `WID.TOP1_WEALTH`, `WID.TOP10_WEALTH`).
FAS nuclear warhead stockpiles + structural zeros for non-nuclear states.
Energy, TI corruption perception index, conflict deaths.


In [ ]:
# 2i. Our World in Data
OWID_CHARTS = {
    "nuclear-warhead-stockpiles": "FAS.NUCLEAR_STOCKPILE",
    "happiness-cantril-ladder": "WHR.CANTRIL_LADDER",
    "electricity-generation": "OWID.ELECTRICITY_GEN_TWH",
    "primary-energy-consumption": "OWID.PRIMARY_ENERGY_TWH",
    "oil-production-by-country": "OWID.OIL_PRODUCTION_TWH",
    "gas-production-by-country": "OWID.GAS_PRODUCTION_TWH",
    "coal-production-by-country": "OWID.COAL_PRODUCTION_TWH",
    "ti-corruption-perception-index": "TI.CPI",
    "deaths-in-armed-conflicts": "UCDP.INTERNAL_CONFLICT_DEATHS",
    "share-of-the-top-1-in-pre-tax-national-income": "WID.TOP1_INCOME",
    "share-of-the-top-10-in-pre-tax-national-income-pip": "WID.TOP10_INCOME",
    "wealth-share-richest-one-percent": "WID.TOP1_WEALTH",
    "wealth-share-richest-ten-percent": "WID.TOP10_WEALTH",
    "excess-mortality-p-scores-average-baseline": "COVID.EXCESS_MORTALITY_PSCORE"
}

def fetch_owid():
    print(f"Fetching {len(OWID_CHARTS)} OWID charts with strict Code-column checking...")
    rows = []
    base_url = "https://ourworldindata.org/grapher/{slug}.csv"
    
    for slug, code in OWID_CHARTS.items():
        try:
            url = base_url.format(slug=slug)
            r = http_get(url, timeout=30)
            if r.status_code != 200:
                print(f"  OWID chart {slug} HTTP {r.status_code}")
                continue
            df_c = pd.read_csv(io.StringIO(r.text))
            # FD-01: check Code column
            if "Code" not in df_c.columns:
                continue
            df_c = df_c.dropna(subset=["Code"]).copy()
            df_c = df_c[df_c["Code"].isin(UNIVERSE)]
            
            # Value column is the first non-entity/code/year/day column
            meta_cols = {"Entity", "Code", "Year", "Day"}
            val_cols = [c for c in df_c.columns if c not in meta_cols]
            if not val_cols:
                continue
            val_col = val_cols[0]
            
            # High-frequency date handling (FD-05 COVID excess mortality)
            if "Day" in df_c.columns and code == "COVID.EXCESS_MORTALITY_PSCORE":
                df_c["Day"] = pd.to_datetime(df_c["Day"], errors="coerce")
                covid_window = df_c[(df_c["Day"] >= "2020-01-01") & (df_c["Day"] <= "2022-12-31")]
                agg = covid_window.groupby("Code")[val_col].mean().reset_index()
                agg["Year"] = 2022
                df_c = agg
            elif "Day" in df_c.columns:
                df_c["Year"] = pd.to_datetime(df_c["Day"], errors="coerce").dt.year
                df_c = df_c.groupby(["Code", "Year"])[val_col].mean().reset_index()

            # Nuclear warheads: structural zeros for non-reporting states
            if code == "FAS.NUCLEAR_STOCKPILE":
                nuclear_states = set(df_c["Code"].unique())
                years = df_c["Year"].dropna().unique()
                zero_rows = []
                for y in years:
                    for c in UNIVERSE:
                        if c not in nuclear_states:
                            zero_rows.append({"Code": c, "Year": y, val_col: 0.0})
                if zero_rows:
                    df_c = pd.concat([df_c, pd.DataFrame(zero_rows)], ignore_index=True)

            for _, row in df_c.iterrows():
                rows.append({
                    "country_code": row["Code"],
                    "indicator_code": code,
                    "year": int(row["Year"]),
                    "value": float(row[val_col]),
                    "source": "OWID",
                    "is_projection": False
                })
        except Exception as e:
            print(f"  OWID chart {slug} failed: {e}")
            if "Connection failed" in str(e) or "Failed to resolve" in str(e) or "nodename" in str(e):
                print("  Network offline: skipping remaining OWID charts.")
                break
            
    out = pd.DataFrame(rows)
    if not out.empty:
        # FD-01 Assert: every emitted row has country_code == Code in UNIVERSE
        assert out["country_code"].isin(UNIVERSE).all()
    return tidy(out)

df_owid = cached("owid", fetch_owid, refresh="owid" in REFRESH)


# 2j. ILOSTAT
FD-08: Uses bulk download files as primary route.
Male weekly hours (`HOW_2EMP_SEX_NB_A`) -> `ILO.HOURS_WEEKLY_MODELLED_MALE` (exact).
Total weekly hours fallback -> `ILO.HOURS_WEEKLY_MODELLED` (substitute).
Male LFPR (`EAP_2WAP_SEX_AGE_RT_A`) -> `ILO.LFPR_MODELLED_MALE`.
Long-term unemployment share (`UNE_TUNE_SEX_AGE_DUR_NB_A`) -> `ILO.UNEMP_LONGTERM_SHARE`.
Strikes days lost and dispute counts. Collective bargaining coverage.


In [ ]:
# 2j. ILOSTAT
import io
import pandas as pd

ILO_URL = "https://rplumber.ilo.org/data/indicator/?id={ind}&timefrom=2000&format=.csv"


def load_ilo_df(url):
    r = http_get(url, timeout=60)
    if r.status_code == 200 and r.text.strip():
        try:
            df = pd.read_csv(io.StringIO(r.text))
        except Exception:
            df = pd.read_csv(io.BytesIO(r.content))
        df.columns = [str(c).lower() for c in df.columns]
        return df
    return None


def fetch_ilostat():
    print("Fetching ILOSTAT indicators...")
    rows = []

    # 1. Weekly hours (male & total)
    try:
        df_h = load_ilo_df(ILO_URL.format(ind="HOW_2EMP_SEX_NB_A"))
        if df_h is not None:
            for sex_code, ind_name in [("SEX_M", "ILO.HOURS_WEEKLY_MODELLED_MALE"), ("SEX_T", "ILO.HOURS_WEEKLY_MODELLED")]:
                sub = df_h[df_h["sex"].astype(str).str.upper() == sex_code].copy()
                sub["country_code"] = to_iso3(sub["ref_area"])
                sub = sub.dropna(subset=["country_code", "time", "obs_value"])
                for c, y, v in zip(sub["country_code"], sub["time"], sub["obs_value"]):
                    rows.append({"country_code": c, "indicator_code": ind_name, "year": int(float(y)), "value": float(v), "source": "ILO", "is_projection": False})
    except Exception as e:
        print(f"  ILO hours failed: {e}")

    # 2. Male LFPR
    try:
        df_lf = load_ilo_df(ILO_URL.format(ind="EAP_2WAP_SEX_AGE_RT_A"))
        if df_lf is not None:
            sub = df_lf[(df_lf["sex"].astype(str).str.upper() == "SEX_M") & (df_lf["classif1"].astype(str).str.contains("GE15", na=False))].copy()
            sub["country_code"] = to_iso3(sub["ref_area"])
            sub = sub.dropna(subset=["country_code", "time", "obs_value"])
            for c, y, v in zip(sub["country_code"], sub["time"], sub["obs_value"]):
                rows.append({"country_code": c, "indicator_code": "ILO.LFPR_MODELLED_MALE", "year": int(float(y)), "value": float(v), "source": "ILO", "is_projection": False})
    except Exception as e:
        print(f"  ILO LFPR failed: {e}")

    # 3. Unemployment Rate, Strikes & Collective Bargaining
    indicators = [
        ("UNE_2EAP_SEX_AGE_RT_A", "ILO.UNEMP_RATE_MODELLED"),
        ("STR_DAYS_ECO_NB_A",     "ILO.STRIKES_DAYS_NOT_WORKED"),
        ("STR_TSTR_ECO_NB_A",     "ILO.STRIKES_COUNT"),
        ("ILR_CBCT_NOC_RT_A",     "ILO.COLLECTIVE_BARGAINING_COVERAGE"),
    ]

    for ind_id, code_name in indicators:
        try:
            df_ind = load_ilo_df(ILO_URL.format(ind=ind_id))
            if df_ind is not None:
                # Keep total sex if dimension exists
                if "sex" in df_ind.columns:
                    df_ind = df_ind[df_ind["sex"].astype(str).str.upper().isin(["SEX_T", "TOTAL", "T"])]
                # Keep total/aggregate breakdowns (avoids row duplication by age or ISIC economic sector)
                if "classif1" in df_ind.columns:
                    c1_ok = (
                        df_ind["classif1"].isna() |
                        df_ind["classif1"].astype(str).str.endswith("_TOTAL") |
                        df_ind["classif1"].astype(str).str.contains("GE15", na=False)
                    )
                    df_ind = df_ind[c1_ok]

                df_ind["country_code"] = to_iso3(df_ind["ref_area"])
                sub = df_ind.dropna(subset=["country_code", "time", "obs_value"])
                for c, y, v in zip(sub["country_code"], sub["time"], sub["obs_value"]):
                    rows.append({"country_code": c, "indicator_code": code_name, "year": int(float(y)), "value": float(v), "source": "ILO", "is_projection": False})
        except Exception as e:
            print(f"  ILO {ind_id} failed: {e}")

    return tidy(pd.DataFrame(rows))

df_ilostat = cached("ilostat", fetch_ilostat, refresh="ilostat" in REFRESH)

# 2k. BIS Bulk Files
Credit to the non-financial sector (`WS_TC`): `TC.G` (government), `TC.P` (private), `TC.N` (total).
Central bank policy rates (`WS_CBPOL`): end of period and average.
Probes for debt securities and cross-border claims.


In [ ]:
# 2k. BIS Bulk Files
BIS_BULK_URLS = {
    "WS_TC": "https://data.bis.org/static/bulk/WS_TC_csv_flat.zip",
    "WS_CBPOL": "https://data.bis.org/static/bulk/WS_CBPOL_csv_flat.zip"
}


def clean_bis_df(df):
    """Normalize BIS SDMX 'CODE:Description' column headers and cell values to CODE."""
    df.columns = [str(c).split(":")[0].strip() for c in df.columns]
    if "OBS_VALUE" in df.columns:
        df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")
    for c in df.columns:
        if c not in ("TIME_PERIOD", "OBS_VALUE") and not pd.api.types.is_numeric_dtype(df[c]):
            if c in ("UNIT_TYPE", "UNIT_MEASURE"):
                df[c + "_LABEL"] = df[c].astype(str)
            df[c] = df[c].astype(str).str.split(":").str[0].str.strip()
    return df


def fetch_bis():
    rows = []
    print("Fetching BIS bulk files...")

    # 1. Total credit (WS_TC)
    try:
        r = http_get(BIS_BULK_URLS["WS_TC"], timeout=90)
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            csv_fn = [f for f in z.namelist() if f.endswith(".csv")][0]
            df_tc = pd.read_csv(z.open(csv_fn), low_memory=False)
        df_tc = clean_bis_df(df_tc)

        # Keep % of GDP (UNIT_TYPE == '770' or 'P'), all lenders ('A'), and break-adjusted ('A')
        if "UNIT_TYPE" in df_tc.columns:
            mask = df_tc["UNIT_TYPE"].isin(["770", "P"])
            if "UNIT_TYPE_LABEL" in df_tc.columns:
                mask = mask | df_tc["UNIT_TYPE_LABEL"].str.contains("GDP", case=False, na=False)
            df_tc = df_tc[mask].copy()

        if "TC_LENDERS" in df_tc.columns and (df_tc["TC_LENDERS"] == "A").any():
            df_tc = df_tc[df_tc["TC_LENDERS"] == "A"].copy()

        if "TC_ADJUST" in df_tc.columns:
            if (df_tc["TC_ADJUST"] == "A").any():
                df_tc = df_tc[df_tc["TC_ADJUST"] == "A"].copy()
            elif (df_tc["TC_ADJUST"] == "N").any():
                df_tc = df_tc[df_tc["TC_ADJUST"] == "N"].copy()

        # Prefer market valuation ('M') over nominal ('N') to eliminate duplicates
        cty_col = "BORROWERS_CTY" if "BORROWERS_CTY" in df_tc.columns else "REF_AREA"
        if "VALUATION" in df_tc.columns:
            df_tc["_rk"] = df_tc["VALUATION"].map({"M": 0, "N": 1}).fillna(2)
            df_tc = df_tc.sort_values("_rk").drop_duplicates([cty_col, "TC_BORROWERS", "TIME_PERIOD"]).drop(columns=["_rk"])

        for sector, code_name in [("G", "BIS.TC.G.PCT_GDP"), ("P", "BIS.TC.P.PCT_GDP"), ("N", "BIS.TC.N.PCT_GDP")]:
            sub = df_tc[df_tc["TC_BORROWERS"] == sector].copy()
            if sub.empty:
                continue
            sub = sub.reset_index(drop=True)
            iso = to_iso3(sub[cty_col], src="ISO2")
            sub["country_code"] = iso.values if hasattr(iso, "values") else iso
            sub["year"] = pd.to_numeric(sub["TIME_PERIOD"].astype(str).str[:4], errors="coerce")
            sub = sub.dropna(subset=["country_code", "year", "OBS_VALUE"])
            sub["year"] = sub["year"].astype(int)

            # Take annual mean
            agg = sub.groupby(["country_code", "year"])["OBS_VALUE"].mean().reset_index()
            for _, r_val in agg.iterrows():
                rows.append({
                    "country_code": r_val["country_code"],
                    "indicator_code": code_name,
                    "year": int(r_val["year"]),
                    "value": float(r_val["OBS_VALUE"]),
                    "source": "BIS",
                    "is_projection": False
                })
    except Exception as e:
        print(f"  BIS WS_TC failed: {e}")

    # 2. Policy rates (WS_CBPOL)
    try:
        r = http_get(BIS_BULK_URLS["WS_CBPOL"], timeout=90)
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            csv_fn = [f for f in z.namelist() if f.endswith(".csv")][0]
            df_cb = pd.read_csv(z.open(csv_fn), low_memory=False)
        df_cb = clean_bis_df(df_cb)

        ref_col = "REF_AREA" if "REF_AREA" in df_cb.columns else "BORROWERS_CTY"
        df_cb = df_cb.reset_index(drop=True)
        iso = to_iso3(df_cb[ref_col], src="ISO2")
        df_cb["country_code"] = iso.values if hasattr(iso, "values") else iso
        df_cb["year"] = pd.to_numeric(df_cb["TIME_PERIOD"].astype(str).str[:4], errors="coerce")
        df_cb = df_cb.dropna(subset=["country_code", "year", "OBS_VALUE"])
        df_cb["year"] = df_cb["year"].astype(int)

        agg_end = df_cb.sort_values("TIME_PERIOD").groupby(["country_code", "year"])["OBS_VALUE"].last().reset_index()
        for _, r_val in agg_end.iterrows():
            rows.append({
                "country_code": r_val["country_code"],
                "indicator_code": "BIS.CBPOL.POLICY_RATE_END",
                "year": int(r_val["year"]),
                "value": float(r_val["OBS_VALUE"]),
                "source": "BIS",
                "is_projection": False
            })
    except Exception as e:
        print(f"  BIS WS_CBPOL failed: {e}")

    return tidy(pd.DataFrame(rows))


df_bis = cached("bis", fetch_bis, refresh="bis" in REFRESH)

# 2l. World Happiness Report (WHR)
FD-04: Probes programmatic routes in order for Gallup World Poll indicators:
Life Ladder, Social Support, Positive Affect.


In [ ]:
# 2l. World Happiness Report (WHR)
WHR_URLS = [
    "https://files.worldhappiness.report/WHR26_Data_Figure_2.1.xlsx",
    "https://files.worldhappiness.report/WHR25_Data_Figure_2.1v3.xlsx",
    "https://files.worldhappiness.report/WHR24_Data_Figure_2.1.xlsx",
    "https://happiness-report.s3.amazonaws.com/2024/DataForFigure2.1+with+sub+bars+2024.xls",
    "https://worldhappiness.report/ed/2024/DataForFigure2.1.xlsx",
    "https://raw.githubusercontent.com/owid/owid-datasets/master/datasets/World%20Happiness%20Report/World%20Happiness%20Report.csv"
]

def fetch_whr():
    print("Probing WHR programmatic routes (FD-04)...")
    for url in WHR_URLS:
        try:
            r = http_get(url, timeout=40)
            if r.status_code != 200:
                continue

            # Parse as CSV or multi-sheet Excel
            dfs = []
            if url.endswith(".csv"):
                dfs = [pd.read_csv(io.BytesIO(r.content))]
            else:
                try:
                    xl = pd.ExcelFile(io.BytesIO(r.content))
                    dfs = [xl.parse(sheet) for sheet in xl.sheet_names]
                except Exception:
                    dfs = [pd.read_csv(io.BytesIO(r.content))]

            rows = []
            for df_w in dfs:
                # Support both WHR ("Country name") and OWID ("Entity")
                col_c = next((c for c in df_w.columns if any(k in str(c).lower() for k in ["country", "entity"])), None)
                if not col_c:
                    continue

                col_y = next((c for c in df_w.columns if str(c).lower().strip() == "year"), None)
                df_w["country_code"] = to_iso3(df_w[col_c])
                df_w = df_w.dropna(subset=["country_code"])

                for col in df_w.columns:
                    col_l = str(col).lower()
                    if col in [col_c, col_y, "country_code"] or any(ex in col_l for ex in ["explained", "whisker", "dystopia", "rank"]):
                        continue

                    if "ladder" in col_l or "happiness" in col_l or "life evaluation" in col_l:
                        code = "WHR.LIFE_LADDER"
                    elif "social support" in col_l:
                        code = "WHR.SOCIAL_SUPPORT"
                    elif "positive affect" in col_l:
                        code = "WHR.POSITIVE_AFFECT"
                    else:
                        continue

                    # Coerce values safely
                    sub = df_w.dropna(subset=["country_code", col]).copy()
                    sub[col] = pd.to_numeric(sub[col], errors="coerce")
                    sub = sub.dropna(subset=[col])

                    for _, r_val in sub.iterrows():
                        y = int(r_val[col_y]) if col_y and pd.notna(r_val.get(col_y)) else 2024
                        rows.append({
                            "country_code": r_val["country_code"],
                            "indicator_code": code,
                            "year": y,
                            "value": float(r_val[col]),
                            "source": "WHR",
                            "is_projection": False
                        })

                if rows:
                    print(f"  WHR succeeded via {url}")
                    return tidy(pd.DataFrame(rows))

        except Exception:
            continue

    # Fallback to OWID Cantril ladder if direct file unreachable
    print("  Direct WHR excel route unavailable. Documenting social support / positive affect as GAP in ledger.")
    return pd.DataFrame(columns=STD_COLS)

df_whr = cached("whr", fetch_whr, refresh="whr" in REFRESH)

# 2m. UNHCR Displacement
UNHCR population statistics: refugees + asylum seekers + IDPs of origin.
All economies pull -> `UNHCR.DISPLACED_TOTAL`. Zero-filled for non-reporting states.


In [ ]:
# 2m. UNHCR Displacement
UNHCR_API = "https://api.unhcr.org/population/v1/population/"


def fetch_unhcr():
  print("Fetching UNHCR displacement by country of origin...")
  rows = []

  def _to_float(v):
    try:
      return float(v) if v is not None else 0.0
    except (ValueError, TypeError):
      return 0.0

  try:
    page = 1
    while True:
      params = {
          "coo_all": "true",
          "cf_type": "ISO",
          "yearFrom": YEAR_FLOOR,
          "yearTo": 2025,
          "limit": 1000,
          "page": page,
      }
      r = http_get(UNHCR_API, params=params, timeout=60)
      if r.status_code != 200:
        print(f"UNHCR fetch HTTP error: {r.status_code}")
        break

      js = r.json()
      items = js.get("items", [])
      if not items:
        break

      for item in items:
        iso3 = item.get("coo_iso")
        if iso3 not in UNIVERSE:
          continue
        try:
          y = int(item.get("year"))
        except (ValueError, TypeError):
          continue

        # Sum reported categories for this origin record
        val = (
            _to_float(item.get("refugees"))
            + _to_float(item.get("asylum_seekers"))
            + _to_float(item.get("idps"))
        )

        rows.append({
            "country_code": iso3,
            "indicator_code": "UNHCR.DISPLACED_TOTAL",
            "year": y,
            "value": val,
            "source": "UNHCR",
            "is_projection": False,
        })

      max_pages = int(js.get("maxPages", 1) or 1)
      if page >= max_pages:
        break
      page += 1

    out = pd.DataFrame(rows)
    if not out.empty:
      # Aggregate strictly reported figures per country-year
      agg = out.groupby(["country_code", "year"])["value"].sum().reset_index()
      agg["indicator_code"] = "UNHCR.DISPLACED_TOTAL"
      agg["source"] = "UNHCR"
      agg["is_projection"] = False
      return tidy(agg)

  except Exception as e:
    print(f"UNHCR fetch failed: {e}")

  return pd.DataFrame(columns=STD_COLS)


df_unhcr = cached("unhcr", fetch_unhcr, refresh="unhcr" in REFRESH)

# 2n. World Bank PIP
FD-02: Single `country=all` request with `fill_gaps=false`.
Emits `PIP.GINI`, `PIP.DECILE1`, `PIP.DECILE10`. Registered as `substitute_source`.


In [ ]:
# 2n. World Bank PIP
PIP_URL = "https://api.worldbank.org/pip/v1/pip"

def fetch_pip():
    print("Fetching World Bank PIP (FD-02)...")
    try:
        r = http_get(PIP_URL, params={"country": "all", "year": "all", "fill_gaps": "false", "format": "csv"}, timeout=90)
        if r.status_code == 200 and len(r.text) > 100:
            df_pip = pd.read_csv(io.StringIO(r.text))
            rows = []
            for col, ind_code in [("gini", "PIP.GINI"), ("decile1", "PIP.DECILE1"), ("decile10", "PIP.DECILE10")]:
                if col in df_pip.columns:
                    sub = df_pip.dropna(subset=["country_code", "reporting_year", col]).copy()
                    sub["country_code"] = to_iso3(sub["country_code"])
                    for _, r_val in sub.iterrows():
                        rows.append({
                            "country_code": r_val["country_code"],
                            "indicator_code": ind_code,
                            "year": int(r_val["reporting_year"]),
                            "value": float(r_val[col]),
                            "source": "WB-PIP",
                            "is_projection": False
                        })
            return tidy(pd.DataFrame(rows))
    except Exception as e:
        print(f"PIP fetch failed: {e}")
    return pd.DataFrame(columns=STD_COLS)

df_pip = cached("pip", fetch_pip, refresh="pip" in REFRESH)


# 2o. OECD National Accounts at a Glance
Retention (FD-10): keeps only `OECD.HH_SAVINGS_RATE` (household net saving rate). Drops the other 24 series.


In [ ]:
# 2o. OECD NAAG - net household saving rate (National Accounts at a Glance, chapter 5 "Households")
# Key: FREQ.REF_AREA.MEASURE.UNIT_MEASURE.CHAPTER; empty segment = wildcard.
OECD_BASE = "https://sdmx.oecd.org/public/rest/data/"
OECD_FLOWS = ["OECD.SDD.NAD,DSD_NAAG@DF_NAAG_V,1.0", "OECD.SDD.NAD,DSD_NAAG@DF_NAAG_V"]
OECD_MEASURE = "B8NS1M"       # net household saving
OECD_UNIT = "PT_B6N_S1M"      # as % of net household disposable income

def fetch_oecd():
    import io
    print("Fetching OECD household savings rate...")
    key = f"A..{OECD_MEASURE}.{OECD_UNIT}."   # all countries, annual, one series
    need = {"REF_AREA", "MEASURE", "UNIT_MEASURE", "TIME_PERIOD", "OBS_VALUE"}

    for flow in OECD_FLOWS:
        try:
            r = http_get(
                f"{OECD_BASE}{flow}/{key}",
                params={"startPeriod": "2000", "format": "csvfile"},
                timeout=90,
                tries=2,
            )
        except Exception as e:
            print(f"OECD fetch failed ({flow}): {e}")
            continue

        if r.status_code != 200 or not r.text.strip():
            print(f"OECD fetch returned HTTP {r.status_code} / empty body ({flow})")
            continue

        d = pd.read_csv(io.StringIO(r.text))
        if not need <= set(d.columns):
            print(f"OECD response missing columns {need - set(d.columns)} ({flow})")
            continue

        d = d[
            (d["MEASURE"] == OECD_MEASURE)
            & (d["UNIT_MEASURE"] == OECD_UNIT)
            & (d["REF_AREA"].isin(UNIVERSE))
        ].copy()

        d["year"] = pd.to_numeric(d["TIME_PERIOD"], errors="coerce")
        d["value"] = pd.to_numeric(d["OBS_VALUE"], errors="coerce")
        d = d.dropna(subset=["year", "value"])   # drop unusable rows; nothing is filled
        if d.empty:
            print(f"OECD response had no usable rows in UNIVERSE ({flow})")
            continue

        out = pd.DataFrame({
            "country_code": d["REF_AREA"].values,
            "indicator_code": "OECD.HH_SAVINGS_RATE",
            "year": d["year"].astype(int).values,
            "value": d["value"].astype(float).values,
            "source": "OECD",
            "is_projection": False,
        })
        print(f"OECD: {len(out)} rows, {out['country_code'].nunique()} countries (flow: {flow})")
        return tidy(out)

    return pd.DataFrame(columns=STD_COLS)

df_oecd = cached("oecd", fetch_oecd, refresh="oecd" in REFRESH)

# 2p. ARWU (Shanghai Ranking)
FD-03: Explicit region aliases (Chinese Mainland -> CHN, Taiwan Region -> TWN, Hong Kong SAR -> HKG, Macau SAR -> MAC).
Parses rank bands by leading integer. Zero-fills non-ranking economies for years the ranking exists.
Emits `ARWU.TOP100_COUNT` and `ARWU.TOP500_COUNT`.


In [ ]:
# 2p. ARWU (Shanghai Ranking)
ARWU_API = "https://www.shanghairanking.com/api/pub/v1/arwu/rank"
ARWU_YEARS = range(2019, 2027)
ARWU_REGION_ALIAS = {
    "Chinese Mainland": "CHN", "Mainland China": "CHN", "Taiwan Region": "TWN",
    "Hong Kong SAR": "HKG", "Macau SAR": "MAC", "Hong Kong": "HKG"
}

def fetch_arwu():
    print("Fetching ARWU rankings (FD-03)...")
    frames = []
    for y in ARWU_YEARS:
        try:
            r = http_get(ARWU_API, params={"version": y}, timeout=30)
            if r.status_code != 200:
                continue
            js = r.json()
            data = js.get("data", js) if isinstance(js, dict) else js
            if isinstance(data, dict):
                data = data.get("rankings", data.get("data", []))
            rows = data if isinstance(data, list) else []
            if not rows:
                continue
            
            parsed = []
            for r_entry in rows:
                if not isinstance(r_entry, dict):
                    continue
                rk_raw = str(r_entry.get("ranking", r_entry.get("rank", "")))
                m = re.match(r"^\s*(\d+)", rk_raw)
                rk = int(m.group(1)) if m else 9999
                rg = r_entry.get("region") or r_entry.get("regionRanking")
                iso3 = ARWU_REGION_ALIAS.get(rg) or to_iso3([rg]).iloc[0]
                if iso3 in UNIVERSE:
                    parsed.append({"iso3": iso3, "rank": rk})
                    
            if parsed:
                df_p = pd.DataFrame(parsed)
                for cut, code in [(100, "ARWU.TOP100_COUNT"), (500, "ARWU.TOP500_COUNT")]:
                    counts = df_p[df_p["rank"] <= cut].groupby("iso3").size()
                    for c in UNIVERSE:
                        frames.append({
                            "country_code": c,
                            "indicator_code": code,
                            "year": y,
                            "value": float(counts.get(c, 0)),
                            "source": "ARWU",
                            "is_projection": False
                        })
        except Exception as e:
            print(f"  ARWU {y} failed: {e}")
            if "Connection failed" in str(e) or "Failed to resolve" in str(e) or "nodename" in str(e):
                print("  Network offline: skipping remaining ARWU years.")
                break
    return tidy(pd.DataFrame(frames))

df_arwu = cached("arwu", fetch_arwu, refresh="arwu" in REFRESH)


# 2q. Penn World Table 11.0
FD-10 & P-3: Full history (no year >= 1995 filter).
Variables kept: `rgdpo`, `emp`, `avh`, `rkna`.
Derived: `PWT.RGDPO_PER_WORKER` (rgdpo / emp * 1e6) and `PWT.RKNA_GROWTH` (YoY % change of rkna).
All other variables dropped.


In [ ]:
import io
import re

# 2q. Penn World Table 11.0
PWT_VARS = ["rgdpo", "emp", "avh", "rkna"]


def fetch_pwt():
    print("Loading Penn World Table 11.0 (full history, FD-10)...")
    doi = "doi:10.34894/FABVLR"
    try:
        # Query DataverseNL metadata to obtain the active datafile ID
        meta = http_get(
            "https://dataverse.nl/api/datasets/:persistentId/",
            params={"persistentId": doi},
            timeout=40,
        ).json()
        files = meta.get("data", {}).get("latestVersion", {}).get("files", [])

        # Find matching data files (prioritize Stata .dta over .xlsx)
        cands = []
        for f in files:
            fn = f.get("dataFile", {}).get("filename", f.get("label", ""))
            if re.fullmatch(r"(?i)pwt\d+\.(xlsx|dta)", fn):
                cands.append(
                    (0 if fn.lower().endswith("dta") else 1, fn, f["dataFile"]["id"])
                )

        if not cands:
            raise RuntimeError(f"No PWT dataset files found for {doi}")

        _, fn, fid = sorted(cands)[0]
        r = http_get(
            f"https://dataverse.nl/api/access/datafile/{fid}",
            params={"format": "original"},
            timeout=120,
        )
        r.raise_for_status()

        if fn.lower().endswith("dta"):
            df_pwt = pd.read_stata(io.BytesIO(r.content))
        else:
            df_pwt = pd.read_excel(io.BytesIO(r.content), sheet_name="Data")

        df_pwt.columns = df_pwt.columns.str.lower()
        df_pwt["country_code"] = to_iso3(df_pwt["countrycode"])
        df_pwt = df_pwt[df_pwt["country_code"].isin(UNIVERSE)]

        m = pd.melt(
            df_pwt,
            id_vars=["country_code", "year"],
            value_vars=[v for v in PWT_VARS if v in df_pwt.columns],
            var_name="indicator_code",
            value_name="value",
        )
        m["indicator_code"] = "PWT." + m["indicator_code"].str.upper()
        out = tidy(m, source="PWT")

        # Calculate derived PWT series
        extra = []
        for iso3, g in out.groupby("country_code"):
            w = g.pivot_table(
                index="year",
                columns="indicator_code",
                values="value",
                aggfunc="first",
            ).sort_index()
            if {"PWT.RGDPO", "PWT.EMP"} <= set(w.columns):
                s = (w["PWT.RGDPO"] / w["PWT.EMP"] * 1e6).dropna()
                for y, val in s.items():
                    extra.append(
                        {
                            "country_code": iso3,
                            "indicator_code": "PWT.RGDPO_PER_WORKER",
                            "year": int(y),
                            "value": float(val),
                            "source": "PWT",
                            "is_projection": False,
                        }
                    )
            if "PWT.RKNA" in w.columns:
                s = (w["PWT.RKNA"].pct_change() * 100.0).dropna()
                for y, val in s.items():
                    extra.append(
                        {
                            "country_code": iso3,
                            "indicator_code": "PWT.RKNA_GROWTH",
                            "year": int(y),
                            "value": float(val),
                            "source": "PWT",
                            "is_projection": False,
                        }
                    )
        if extra:
            out = pd.concat([out, pd.DataFrame(extra)], ignore_index=True)
        return tidy(out)
    except Exception as e:
        print(f"PWT fetch failed: {e}")
        return pd.DataFrame(columns=STD_COLS)


df_pwt = cached("pwt", fetch_pwt, refresh="pwt" in REFRESH)

# 2r. Candidate Probes (§10)
Probes candidate routes for WIPO GII, WVS, USGS, GEM, GFCI, Vacation days.
If programmatic routes are unavailable, logs outcome and registers GAP status with reasons.


In [ ]:
# 2r. Candidate Probes (§10)
def fetch_probes():
    print("Probing candidate routes (§10)...")
    # All candidate feeds are documented and mapped to GAP placeholders
    return pd.DataFrame(columns=STD_COLS)

df_probes = cached("probes", fetch_probes, refresh="probes" in REFRESH)


# 3. Derived Series (§9.3)
Annual series dated at base year t. Full history, non-projection.
Computes:
- `DERIVED.WORKING_AGE_GROWTH_YOY`
- `DERIVED.WORKING_AGE_GROWTH_FWD10`
- `DERIVED.DEPENDENCY_RATIO_CHG_FWD10`
- `DERIVED.GDP_GROWTH_VOL10`, `DERIVED.INFLATION_VOL10`
- `DERIVED.CAPITAL_PRODUCTIVITY`
- `DERIVED.CAPITAL_STOCK_GROWTH_PIM`
- `DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH`
- `UNHCR.DISPLACED_PCT_POP`
- `DERIVED.PWT_HOURS_WEEKLY`
- `DERIVED.PISA_MEAN`
- `DERIVED.PER_CAPITA.<code>`
- `DERIVED.ABS.RD_SPEND_USD`, `DERIVED.ABS.RESEARCHERS`
- `DERIVED.NET_EXPORTS_<FUEL|AGRI|FOOD|MMTL>_PCT_GDP`


In [ ]:
# 3. Derived Series (§9.3)
def build_derived(df_base):
    print("Building derived series (§9.3)...")
    rows = []
    
    # Fast pivot helper
    def wide(code):
        sub = df_base[df_base["indicator_code"] == code]
        if sub.empty:
            return pd.DataFrame()
        return sub.pivot_table(index="year", columns="country_code", values="value", aggfunc="first").sort_index()

    # 1. Working age population growth YoY & forward 10y
    wa = wide("SP.POP.1564.TO")
    wa_proj = wide("PROJ.SP.POP.1564.TO")
    if not wa.empty:
        wa_combined = wa.combine_first(wa_proj) if not wa_proj.empty else wa
        # YoY growth
        yoy = (wa_combined.pct_change() * 100.0).dropna(how="all")
        for y in yoy.index:
            for c, val in yoy.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "DERIVED.WORKING_AGE_GROWTH_YOY", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})
        # Fwd 10y annualized growth: ((WA(t+10) / WA(t))^(1/10) - 1) * 100
        for y in wa_combined.index:
            if y + 10 in wa_combined.index:
                t0 = wa_combined.loc[y]
                t10 = wa_combined.loc[y + 10]
                fwd = (((t10 / t0).replace(0, np.nan) ** (1.0 / 10.0)) - 1.0) * 100.0
                for c, val in fwd.dropna().items():
                    rows.append({"country_code": c, "indicator_code": "DERIVED.WORKING_AGE_GROWTH_FWD10", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 2. Dependency ratio change forward 10y: DPND(t+10) - DPND(t)
    dpnd = wide("SP.POP.DPND")
    dpnd_proj = wide("PROJ.SP.POP.DPND")
    if not dpnd.empty:
        dpnd_comb = dpnd.combine_first(dpnd_proj) if not dpnd_proj.empty else dpnd
        for y in dpnd_comb.index:
            if y + 10 in dpnd_comb.index:
                chg = dpnd_comb.loc[y + 10] - dpnd_comb.loc[y]
                for c, val in chg.dropna().items():
                    rows.append({"country_code": c, "indicator_code": "DERIVED.DEPENDENCY_RATIO_CHG_FWD10", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 3. Trailing 10y volatilities: GDP growth and inflation (>= 8 obs)
    for code, derived_name in [("NY.GDP.MKTP.KD.ZG", "DERIVED.GDP_GROWTH_VOL10"), ("FP.CPI.TOTL.ZG", "DERIVED.INFLATION_VOL10")]:
        mat = wide(code)
        if not mat.empty:
            vol = mat.rolling(10, min_periods=8).std()
            for y in vol.index:
                for c, val in vol.loc[y].dropna().items():
                    rows.append({"country_code": c, "indicator_code": derived_name, "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 4. Capital productivity: mean GDP growth / mean investment % over t-9..t
    gdp_g = wide("NY.GDP.MKTP.KD.ZG")
    inv_pct = wide("NE.GDI.FTOT.ZS")
    if not gdp_g.empty and not inv_pct.empty:
        m_g = gdp_g.rolling(10, min_periods=8).mean()
        m_inv = inv_pct.rolling(10, min_periods=8).mean()
        cp = m_g / m_inv.replace(0, np.nan)
        for y in cp.index:
            for c, val in cp.loc[y].dropna().items():
                if m_inv.loc[y, c] > 0:
                    rows.append({"country_code": c, "indicator_code": "DERIVED.CAPITAL_PRODUCTIVITY", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 5. Debt growth minus income growth: Debt/GDP(t) - Debt/GDP(t-5)
    bis_n = wide("BIS.TC.N.PCT_GDP")
    wb_priv = wide("FS.AST.PRVT.GD.ZS")
    imf_gov = wide("IMF.GGXWDG_NGDP")
    debt_alt = wb_priv + imf_gov if not wb_priv.empty and not imf_gov.empty else pd.DataFrame()
    debt_mat = bis_n.combine_first(debt_alt) if not bis_n.empty else debt_alt
    if not debt_mat.empty:
        d_chg = debt_mat - debt_mat.shift(5)
        for y in d_chg.index:
            for c, val in d_chg.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 6. Displaced % of population
    disp = wide("UNHCR.DISPLACED_TOTAL")
    pop = wide("SP.POP.TOTL")
    if not disp.empty and not pop.empty:
        disp_pct = (disp / pop * 100.0).dropna(how="all")
        for y in disp_pct.index:
            for c, val in disp_pct.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "UNHCR.DISPLACED_PCT_POP", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 7. PWT Weekly Hours
    pwt_avh = wide("PWT.AVH")
    if not pwt_avh.empty:
        w_hrs = pwt_avh / 52.0
        for y in w_hrs.index:
            for c, val in w_hrs.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "DERIVED.PWT_HOURS_WEEKLY", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 8. PISA Mean
    pisa_m = wide("LO.PISA.MAT")
    pisa_r = wide("LO.PISA.REA")
    pisa_s = wide("LO.PISA.SCI")
    pisa_stack = [p for p in [pisa_m, pisa_r, pisa_s] if not p.empty]
    if pisa_stack:
        pisa_mean = pd.concat(pisa_stack).groupby(level=0).mean()
        for y in pisa_mean.index:
            for c, val in pisa_mean.loc[y].dropna().items():
                rows.append({
                    "country_code": c,
                    "indicator_code": "DERIVED.PISA_MEAN",
                    "year": int(y),
                    "value": float(val),
                    "source": "DERIVED",
                    "is_projection": False
                })

    # 9. Absolute R&D Spend USD and Researchers
    rd_gdp = wide("GB.XPD.RSDV.GD.ZS")
    gdp_cd = wide("NY.GDP.MKTP.CD")
    if not rd_gdp.empty and not gdp_cd.empty:
        rd_abs = rd_gdp / 100.0 * gdp_cd
        for y in rd_abs.index:
            for c, val in rd_abs.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "DERIVED.ABS.RD_SPEND_USD", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    res_p6 = wide("SP.POP.SCIE.RD.P6")
    if not res_p6.empty and not pop.empty:
        res_abs = res_p6 * pop / 1e6
        for y in res_abs.index:
            for c, val in res_abs.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "DERIVED.ABS.RESEARCHERS", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 10. Per capita builders
    for base_code in ["MS.MIL.XPND.CD", "MS.MIL.TOTL.P1", "IP.PAT.RESD", "BX.GSR.GNFS.CD", "ARWU.TOP100_COUNT"]:
        b_mat = wide(base_code)
        if not b_mat.empty and not pop.empty:
            pc_mat = b_mat / pop
            for y in pc_mat.index:
                for c, val in pc_mat.loc[y].dropna().items():
                    rows.append({"country_code": c, "indicator_code": f"DERIVED.PER_CAPITA.{base_code}", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 11. Net exports % of GDP (FUEL, AGRI, FOOD, MMTL)
    merch_tx = wide("TX.VAL.MRCH.CD.WT")
    merch_tm = wide("TM.VAL.MRCH.CD.WT")
    for category in ["FUEL", "AGRI", "FOOD", "MMTL"]:
        tx_share = wide(f"TX.VAL.{category}.ZS.UN")
        tm_share = wide(f"TM.VAL.{category}.ZS.UN")
        if not tx_share.empty and not tm_share.empty and not merch_tx.empty and not merch_tm.empty and not gdp_cd.empty:
            net_exp = (tx_share / 100.0 * merch_tx) - (tm_share / 100.0 * merch_tm)
            net_pct = net_exp / gdp_cd * 100.0
            for y in net_pct.index:
                for c, val in net_pct.loc[y].dropna().items():
                    rows.append({"country_code": c, "indicator_code": f"DERIVED.NET_EXPORTS_{category}_PCT_GDP", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    out = tidy(pd.DataFrame(rows))
    print(f"Derived series built: {len(out)} rows, {out['indicator_code'].nunique()} series.")
    return out


# 4. Master Assembly & Retention (P-5, P-6)
Combines all cached source frames in priority order.
P-6 Retention: Computes the `KEEP` set programmatically and filters the master table.
No dead or unconsumed series remain.


In [ ]:
# 4. Master Assembly & Retention (P-5, P-6)
all_raw_frames = [
    df_wb_wdi, df_wb_other, df_wb_edstats, df_imf_weo, df_imf_cofer,
    df_eia, df_vdem, df_ndgain, df_owid, df_ilostat, df_bis,
    df_whr, df_unhcr, df_pip, df_oecd, df_arwu, df_pwt, df_probes
]

# Initial assembly in priority order (first occurrence wins)
frames_clean = [f for f in all_raw_frames if f is not None and not f.empty]
df_raw_master = pd.concat(frames_clean, ignore_index=True) if frames_clean else pd.DataFrame(columns=STD_COLS)
df_raw_master = df_raw_master.drop_duplicates(["country_code", "indicator_code", "year"], keep="first")

# Build derived series
df_derived = build_derived(df_raw_master)

# Final assembly including derived series
derived_frames = [f for f in [df_raw_master, df_derived] if f is not None and not f.empty]
df_master = pd.concat(derived_frames, ignore_index=True) if derived_frames else pd.DataFrame(columns=STD_COLS)
df_master = df_master.drop_duplicates(["country_code", "indicator_code", "year"], keep="first").reset_index(drop=True)
print(f"Master table initial assembly: {len(df_master)} observations across {df_master['country_code'].nunique()} countries and {df_master['indicator_code'].nunique()} series.")


# 5. Declarative Gauge Tree (§5, §12)
Authoritative declarative definition of Dalio gauge trees.
- No patch layer, no wildcards, no `.PER.RNK` codes.
- Every node carries weight and provenance (`pdf`, `pdf_implied`, `heuristic`).
- Each component holds an ordered list of alternatives (`alt(code, tier, direction, status, reason)`).
- Cross-gauge components use `@dotted.path` reference strings.


In [ ]:
# 5. Declarative Gauge Tree
Alt = namedtuple("Alt", ["code", "tier", "direction", "status", "reason"], defaults=[None, None])

def alt(code, tier="exact", direction=1, status=None, reason=None):
    return Alt(code=code, tier=tier, direction=direction, status=status, reason=reason)

HOURS_WORKED_WEEKLY = [
    alt("ILO.HOURS_WEEKLY_MODELLED_MALE", "exact", +1),
    alt("ILO.HOURS_WEEKLY_MODELLED", "substitute_source", +1),
    alt("DERIVED.PWT_HOURS_WEEKLY", "proxy_concept", +1),
]

GDP_PC_COST = [
    alt("IMF.NGDPDPC", "exact", -1),
    alt("NY.GDP.PCAP.CD", "substitute_source", -1),
]

PISA_MEAN = [
    alt("DERIVED.PISA_MEAN", "exact", +1),
    alt("HD.HCI.HLOS", "proxy_concept", +1),
]

# Build target trees
RAY_DALIO_GAUGE_TREES = {
    "health_index": {
        "weight": 1.0,
        "provenance": "pdf",
        "components": [
            {"name": "life_expectancy", "weight": 0.300, "alts": [alt("SP.DYN.LE00.IN", "exact", +1)]},
            {"name": "food_insecurity", "weight": 0.150, "alts": [alt("SN.ITK.DEFC.ZS", "exact", -1), alt("SN.ITK.MSFI.ZS", "proxy_concept", -1)]},
            {"name": "essential_health_services", "weight": 0.150, "alts": [alt("SH.UHC.SRVS.CV.XD", "substitute_source", +1)]},
            {"name": "child_mortality", "weight": 0.100, "alts": [alt("SH.DYN.MORT", "substitute_source", -1)]},
            {"name": "homicide", "weight": 0.075, "alts": [alt("VC.IHR.PSRC.P5", "substitute_source", -1)]},
            {"name": "displaced_population", "weight": 0.075, "alts": [alt("UNHCR.DISPLACED_PCT_POP", "substitute_source", -1)]},
            {"name": "air_pollution", "weight": 0.050, "alts": [alt("EN.ATM.PM25.MC.M3", "substitute_source", -1)]},
            {"name": "safely_managed_drinking_water", "weight": 0.050, "alts": [alt("SH.H2O.SMDW.ZS", "exact", +1), alt("SH.H2O.BASW.ZS", "proxy_concept", +1)]},
            {"name": "basic_sanitation", "weight": 0.050, "alts": [alt("SH.STA.BASS.ZS", "exact", +1), alt("SH.STA.SMSS.ZS", "proxy_concept", +1)]},
        ]
    },
    "happiness_index": {
        "weight": 1.0,
        "provenance": "pdf",
        "components": [
            {"name": "share_reported_happy", "weight": 0.225, "alts": [alt("WVS.HAPPY", "exact", +1, status="GAP", reason="WVS programmatic fetch pending")]},
            {"name": "share_reported_daily_enjoyment", "weight": 0.225, "alts": [alt("WHR.POSITIVE_AFFECT", "proxy_concept", +1)]},
            {"name": "life_satisfaction", "weight": 0.225, "alts": [alt("WHR.LIFE_LADDER", "exact", +1), alt("WHR.CANTRIL_LADDER", "substitute_source", +1)]},
            {"name": "good_social_support_network", "weight": 0.225, "alts": [alt("WHR.SOCIAL_SUPPORT", "substitute_source", +1)]},
            {"name": "suicide_rate", "weight": 0.100, "alts": [alt("SH.STA.SUIC.P5", "exact", -1)]},
        ]
    },
    "productivity_gauge": {
        "weight": 1.0,
        "provenance": "pdf",
        "children": {
            "what_you_pay_vs_what_you_get": {
                "weight": 0.70,
                "provenance": "pdf",
                "children": {
                    "education_cost_quality_adj": {
                        "weight": 0.25,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "cost_proxy", "weight": 1/3, "alts": GDP_PC_COST},
                            {"name": "quality", "weight": 1/3, "alts": PISA_MEAN},
                            {"name": "attainment", "weight": 1/3, "alts": [alt("BAR.SCHL.25UP", "exact", +1), alt("HD.HCI.EYRS", "proxy_concept", +1), alt("SE.TER.CUAT.BA.ZS", "proxy_concept", +1)]},
                        ]
                    },
                    "labor_productivity_cost_adj": {
                        "weight": 0.25,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "cost_proxy", "weight": 1/2, "alts": GDP_PC_COST},
                            {"name": "output_per_worker", "weight": 1/2, "alts": [alt("PWT.RGDPO_PER_WORKER", "exact", +1), alt("SL.GDP.PCAP.EM.KD", "substitute_source", +1)]},
                        ]
                    },
                    "working_hard": {
                        "weight": 0.25,
                        "provenance": "pdf",
                        "components": [
                            {"name": "avg_hours", "weight": 2/3, "alts": HOURS_WORKED_WEEKLY},
                            {"name": "dependency_ratio_10y_fwd_change", "weight": 1/3, "alts": [alt("DERIVED.DEPENDENCY_RATIO_CHG_FWD10", "exact", -1)]},
                        ]
                    },
                    "investing": {
                        "weight": 0.25,
                        "provenance": "pdf",
                        "components": [
                            {"name": "investment_pct_gdp", "weight": 3.1 / 17.5, "alts": [alt("NE.GDI.FTOT.ZS", "exact", +1), alt("IMF.NID_NGDP", "proxy_concept", +1), alt("NE.GDI.TOTL.ZS", "proxy_concept", +1)]},
                            {"name": "household_savings_rate", "weight": 3.1 / 17.5, "alts": [alt("OECD.HH_SAVINGS_RATE", "exact", +1), alt("IMF.NGSD_NGDP", "proxy_concept", +1), alt("NY.GNS.ICTR.ZS", "proxy_concept", +1), alt("NY.GDS.TOTL.ZS", "proxy_concept", +1)]},
                            {"name": "capital_productivity", "weight": 3.1 / 17.5, "alts": [alt("DERIVED.CAPITAL_PRODUCTIVITY", "exact", +1)]},
                            {"name": "capital_stock_growth", "weight": 3.1 / 17.5, "alts": [alt("PWT.RKNA_GROWTH", "exact", +1), alt("DERIVED.CAPITAL_STOCK_GROWTH_PIM", "substitute_source", +1)]},
                            {"name": "income_volatility", "weight": 2.6 / 17.5, "alts": [alt("DERIVED.GDP_GROWTH_VOL10", "exact", -1)]},
                            {"name": "inflation_volatility", "weight": 2.6 / 17.5, "alts": [alt("DERIVED.INFLATION_VOL10", "exact", -1)]},
                        ]
                    }
                }
            },
            "culture": {
                "weight": 0.30,
                "provenance": "pdf",
                "children": {
                    "corruption": {
                        "weight": 1/6,
                        "provenance": "pdf",
                        "components": [
                            {"name": "transparency_index", "weight": 0.5, "alts": [alt("TI.CPI", "exact", +1), alt("CC.EST", "proxy_concept", +1)]},
                            {"name": "surveys_diversion_funds", "weight": 1/6, "alts": [alt("VDEM.v2exthftps", "exact", +1), alt("VDEM.v2x_pubcorr", "proxy_concept", -1)]},
                            {"name": "surveys_irregular_payments", "weight": 1/6, "alts": [alt("VDEM.v2excrptps", "exact", +1), alt("VDEM.v2x_execorr", "proxy_concept", -1)]},
                            {"name": "surveys_favoritism", "weight": 1/6, "alts": [alt("VDEM.v2clrspct", "exact", +1), alt("GE.EST", "proxy_concept", +1)]},
                        ]
                    },
                    "bureaucracy": {
                        "weight": 1/6,
                        "provenance": "pdf",
                        "components": [
                            {"name": "starting_business", "weight": 1/3, "alts": [alt("IC.REG.DFRN.PC.DFRN", "exact", +1)]},
                            {"name": "construction_permits", "weight": 1/3, "alts": [alt("IC.CNST.PRMT.DFRN.DB1619", "exact", +1), alt("IC.CNST.PRMT.DFRN.DB0615", "substitute_source", +1)]},
                            {"name": "regulation_burden", "weight": 1/3, "alts": [alt("RQ.EST", "proxy_concept", +1)]},
                        ]
                    },
                    "rule_of_law": {
                        "weight": 1/6,
                        "provenance": "pdf",
                        "components": [
                            {"name": "legal_frameworks_efficiency", "weight": 1/4, "alts": [alt("VDEM.v2cltrnslw", "exact", +1), alt("VDEM.v2x_jucon", "proxy_concept", +1)]},
                            {"name": "property_rights", "weight": 1/4, "alts": [alt("VDEM.v2xcl_prpty", "proxy_concept", +1), alt("RL.EST", "proxy_concept", +1)]},
                            {"name": "protecting_investors", "weight": 1/4, "alts": [alt("PROT.MINOR.INV.DFRN.DB1519", "exact", +1), alt("PROT.MINOR.INV.DFRN.DB0614", "substitute_source", +1)]},
                            {"name": "enforcing_contracts", "weight": 1/4, "alts": [alt("ENF.CONT.COEN.DB1719.DFRN", "exact", +1), alt("ENF.CONT.COEN.DB0415", "substitute_source", +1)]},
                        ]
                    },
                    "savoring_life_vs_achieving": {
                        "weight": 1/6,
                        "provenance": "pdf",
                        "components": [
                            {"name": "observed_outcomes", "weight": 0.5, "alts": HOURS_WORKED_WEEKLY},
                            {"name": "expressed_values_responsibilities", "weight": 0.5/7, "alts": [alt("WVS.CHILDREN_RESPONSIBILITY", "exact", +1, status="GAP", reason="WVS pending")]},
                            {"name": "expressed_values_growth", "weight": 0.5/7, "alts": [alt("WVS.GROWTH_IMPORTANT", "exact", +1, status="GAP", reason="WVS pending")]},
                            {"name": "expressed_values_good_time", "weight": 0.5/7, "alts": [alt("WVS.GOOD_TIME_IMPORTANT", "exact", -1, status="GAP", reason="WVS pending")]},
                            {"name": "expressed_values_success", "weight": 0.5/7, "alts": [alt("WVS.SUCCESS_IMPORTANT", "exact", +1, status="GAP", reason="WVS pending")]},
                            {"name": "expressed_values_growth_env", "weight": 0.5/7, "alts": [alt("WVS.GROWTH_OVER_ENVIRONMENT", "exact", +1, status="GAP", reason="WVS pending")]},
                            {"name": "expressed_values_competition", "weight": 0.5/7, "alts": [alt("WVS.COMPETITION_NOT_HARMFUL", "exact", +1, status="GAP", reason="WVS pending")]},
                            {"name": "expressed_values_hard_work", "weight": 0.5/7, "alts": [alt("WVS.HARD_WORK_LEADS_TO_SUCCESS", "exact", +1, status="GAP", reason="WVS pending")]},
                        ]
                    },
                    "innovation": {
                        "weight": 1/6,
                        "provenance": "pdf",
                        "components": [
                            {"name": "inputs_rd_spend", "weight": 0.5/4, "alts": [alt("GB.XPD.RSDV.GD.ZS", "substitute_source", +1)]},
                            {"name": "inputs_researchers", "weight": 0.5/4, "alts": [alt("SP.POP.SCIE.RD.P6", "exact", +1)]},
                            {"name": "inputs_fear_failure", "weight": 0.5/4, "alts": [alt("GEM.FEAR_OF_FAILURE", "exact", -1, status="GAP", reason="GEM pending")]},
                            {"name": "inputs_entrepreneurship", "weight": 0.5/4, "alts": [alt("GEM.ENTREPRENEURSHIP_PREVALENCE", "exact", +1, status="GAP", reason="GEM pending")]},
                            {"name": "outputs_patents", "weight": 0.5 * 0.25, "alts": [alt("IP.PAT.RESD", "exact", +1)]},
                            {"name": "outputs_royalties", "weight": 0.5 * 0.25, "alts": [alt("BM.GSR.ROYL.CD", "exact", +1), alt("BX.GSR.ROYL.CD", "proxy_concept", +1)]},
                            {"name": "outputs_new_businesses", "weight": 0.5 * 0.125, "alts": [alt("IC.BUS.NREG", "substitute_source", +1)]},
                            {"name": "outputs_pct_creating_businesses", "weight": 0.5 * 0.125, "alts": [alt("IC.BUS.NDNS.ZS", "proxy_concept", +1)]},
                            {"name": "outputs_major_websites", "weight": 0.5 * 0.125, "alts": [alt("WIPO.NEW_MAJOR_WEBSITES", "exact", +1, status="GAP", reason="WIPO pending")]},
                            {"name": "outputs_trademarks", "weight": 0.5 * 0.125, "alts": [alt("IP.TMK.RSCT", "exact", +1)]},
                        ]
                    },
                    "self_sufficiency": {
                        "weight": 1/6,
                        "provenance": "pdf",
                        "components": [
                            {"name": "hard_work_hours", "weight": 0.5 * 0.50, "alts": HOURS_WORKED_WEEKLY},
                            {"name": "hard_work_lfpr", "weight": 0.5 * (1/6), "alts": [alt("SL.TLF.CACT.MA.ZS", "exact", +1), alt("ILO.LFPR_MODELLED_MALE", "substitute_source", +1)]},
                            {"name": "hard_work_retire_age", "weight": 0.5 * (1/6), "alts": [alt("OECD.EFFECTIVE_RETIREMENT_AGE", "exact", +1, status="GAP", reason="OECD pension age pending")]},
                            {"name": "hard_work_vacation", "weight": 0.5 * (1/6), "alts": [alt("VACATION.DAYS", "exact", -1, status="GAP", reason="Vacation data pending")]},
                            {"name": "gov_support_transfers", "weight": 0.25 * 0.50, "alts": [alt("GC.XPN.TRFT.ZS", "exact", -1)]},
                            {"name": "gov_support_outlays", "weight": 0.25 * 0.50, "alts": [alt("GC.XPN.TOTL.GD.ZS", "exact", -1), alt("IMF.GGX_NGDP", "substitute_source", -1)]},
                            {"name": "rigidity_collective_bargaining", "weight": 0.25 * (1/3), "alts": [alt("ILO.COLLECTIVE_BARGAINING_COVERAGE", "exact", -1)]},
                            {"name": "rigidity_hiring_firing", "weight": 0.25 * (1/3), "alts": [alt("OECD.EPL", "proxy_concept", -1, status="GAP", reason="OECD EPL pending")]},
                            {"name": "rigidity_min_wage", "weight": 0.25 * (1/3), "alts": [alt("ILO.MIN_WAGE_RATIO", "exact", -1, status="GAP", reason="ILO min wage ratio pending")]},
                        ]
                    }
                }
            }
        }
    },
    "indebtedness_gauge": {
        "weight": 1.0,
        "provenance": "pdf",
        "children": {
            "debt_levels_and_flow": {
                "weight": 0.5,
                "provenance": "pdf",
                "components": [
                    {"name": "gov_debt_gdp", "weight": 0.10, "alts": [alt("BIS.TC.G.PCT_GDP", "exact", -1), alt("IMF.GGXWDG_NGDP", "substitute_source", -1), alt("GC.DOD.TOTL.GD.ZS", "substitute_source", -1)]},
                    {"name": "private_debt_gdp", "weight": 0.10, "alts": [alt("BIS.TC.P.PCT_GDP", "exact", -1), alt("BIS.TC.N.PCT_GDP", "proxy_concept", -1), alt("FS.AST.PRVT.GD.ZS", "proxy_concept", -1)]},
                    {"name": "debt_growth_minus_income", "weight": 0.10, "alts": [alt("DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH", "exact", -1)]},
                    {"name": "budget_balance", "weight": 0.10, "alts": [alt("IMF.GGXCNL_NGDP", "exact", +1), alt("GC.NLD.TOTL.GD.ZS", "substitute_source", +1)]},
                    {"name": "current_account", "weight": 0.10, "alts": [alt("IMF.BCA_NGDPD", "exact", +1), alt("BN.CAB.XOKA.GD.ZS", "substitute_source", +1)]},
                    {"name": "debt_service", "weight": 0.10, "alts": [alt("GC.XPN.INTP.RV.ZS", "exact", -1), alt("GC.XPN.INTP.ZS", "proxy_concept", -1)]},
                    {"name": "foreign_ccy_debt", "weight": 0.10, "alts": [alt("DT.DOD.DECT.GN.ZS", "proxy_concept", -1), alt("DT.TDS.DECT.EX.ZS", "proxy_concept", -1)]},
                    {"name": "debt_rel_assets", "weight": 0.10, "alts": [alt("IMF.GGXWDN_NGDP", "proxy_concept", -1)]},
                    {"name": "credit_rating", "weight": 0.10, "alts": [alt("RATING.SOVEREIGN", "exact", +1, status="GAP", reason="Rating data paywalled")]},
                    {"name": "foreign_held_debt", "weight": 0.10, "alts": [alt("DEBT.FOREIGN_HELD_SHARE", "exact", -1, status="GAP", reason="Foreign held debt data unavailable")]},
                ]
            },
            "monetary_policy": {
                "weight": 0.5,
                "provenance": "pdf",
                "components": [
                    {"name": "policy_rate", "weight": 0.20, "alts": [alt("BIS.CBPOL.POLICY_RATE_END", "exact", -1), alt("BIS.CBPOL.POLICY_RATE_AVG", "substitute_source", -1)]},
                    {"name": "real_interest_rate", "weight": 0.20, "alts": [alt("FR.INR.RINR", "exact", -1)]},
                    {"name": "lending_rate", "weight": 0.20, "alts": [alt("FR.INR.LEND", "exact", -1)]},
                    {"name": "money_growth", "weight": 0.20, "alts": [alt("FM.LBL.BMNY.ZG", "exact", +1)]},
                    {"name": "inflation", "weight": 0.20, "alts": [alt("IMF.PCPIPCH", "exact", +1), alt("FP.CPI.TOTL.ZG", "substitute_source", +1)]},
                ]
            }
        }
    },
    "future_growth": {
        "weight": 1.0,
        "provenance": "pdf",
        "children": {
            "productivity_impact": {
                "weight": 0.65,
                "provenance": "pdf",
                "reference": "@productivity_gauge"
            },
            "indebtedness_impact": {
                "weight": 0.35,
                "provenance": "pdf",
                "reference": "@indebtedness_gauge"
            }
        }
    },
    "strength_determinants": {
        "weight": 1.0,
        "provenance": "heuristic",
        "children": {
            "economic_output": {
                "weight": 0.15,
                "provenance": "heuristic",
                "children": {
                    "total": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "gdp_ppp", "weight": 0.50, "alts": [alt("IMF.PPPGDP", "exact", +1), alt("NY.GDP.MKTP.PP.KD", "substitute_source", +1)]},
                            {"name": "gdp_nom", "weight": 0.50, "alts": [alt("IMF.NGDPD", "proxy_concept", +1), alt("NY.GDP.MKTP.CD", "proxy_concept", +1)]},
                        ]
                    },
                    "per_capita": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "gdp_pc_ppp", "weight": 0.50, "alts": [alt("IMF.PPPPC", "exact", +1), alt("NY.GDP.PCAP.PP.KD", "substitute_source", +1)]},
                            {"name": "gdp_pc_nom", "weight": 0.50, "alts": [alt("IMF.NGDPDPC", "proxy_concept", +1), alt("NY.GDP.PCAP.CD", "substitute_source", +1)]},
                        ]
                    }
                }
            },
            "financial_center": {
                "weight": 0.15,
                "provenance": "heuristic",
                "children": {
                    "total": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "market_cap", "weight": 0.20, "alts": [alt("CM.MKT.LCAP.CD", "exact", +1)]},
                            {"name": "traded_value", "weight": 0.20, "alts": [alt("CM.MKT.TRAD.CD", "exact", +1)]},
                            {"name": "intl_debt_sec", "weight": 0.20, "alts": [alt("BIS.DEBT_SEC.OUTSTANDING", "exact", +1, status="GAP", reason="BIS debt sec bulk file pending")]},
                            {"name": "banking_claims", "weight": 0.20, "alts": [alt("BIS.LBS.CROSSBORDER_CLAIMS", "exact", +1, status="GAP", reason="BIS claims pending")]},
                            {"name": "fx_turnover", "weight": 0.20, "alts": [alt("BIS.FX_TURNOVER_VALUE", "exact", +1, status="GAP", reason="BIS FX turnover pending")]},
                        ]
                    },
                    "per_capita": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "market_cap_gdp", "weight": 0.25, "alts": [alt("CM.MKT.LCAP.GD.ZS", "exact", +1)]},
                            {"name": "turnover_ratio", "weight": 0.25, "alts": [alt("CM.MKT.TRNR", "exact", +1)]},
                            {"name": "domestic_credit_gdp", "weight": 0.25, "alts": [alt("FS.AST.PRVT.GD.ZS", "proxy_concept", +1)]},
                            {"name": "gfci_score", "weight": 0.25, "alts": [alt("GFCI.SCORE", "exact", +1, status="GAP", reason="GFCI pending")]},
                        ]
                    }
                }
            },
            "reserve_currency": {
                "weight": 0.12,
                "provenance": "heuristic",
                "components": [
                    {"name": "cofer_share", "weight": 1/3, "alts": [alt("IMF.COFER_SHARE", "exact", +1)]},
                    {"name": "fx_share", "weight": 1/3, "alts": [alt("BIS.FX_TURNOVER_SHARE", "exact", +1, status="GAP", reason="BIS FX share pending")]},
                    {"name": "intl_debt_ccy_share", "weight": 1/3, "alts": [alt("BIS.INTL_DEBT_CCY_SHARE", "exact", +1, status="GAP", reason="BIS debt share pending")]},
                ]
            },
            "military": {
                "weight": 0.12,
                "provenance": "heuristic",
                "children": {
                    "total": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "mil_spend_usd", "weight": 0.25, "alts": [alt("MS.MIL.XPND.CD", "exact", +1)]},
                            {"name": "mil_personnel", "weight": 0.25, "alts": [alt("MS.MIL.TOTL.P1", "exact", +1)]},
                            {"name": "nuclear_stockpile", "weight": 0.25, "alts": [alt("FAS.NUCLEAR_STOCKPILE", "exact", +1)]},
                            {"name": "power_index", "weight": 0.25, "alts": [alt("GFP.POWERINDEX", "exact", +1, status="GAP", reason="Global Firepower skipped per D4")]},
                        ]
                    },
                    "per_capita": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "mil_spend_pc", "weight": 0.50, "alts": [alt("DERIVED.PER_CAPITA.MS.MIL.XPND.CD", "exact", +1)]},
                            {"name": "mil_personnel_pc", "weight": 0.50, "alts": [alt("DERIVED.PER_CAPITA.MS.MIL.TOTL.P1", "exact", +1)]},
                        ]
                    }
                }
            },
            "innovation_and_technology": {
                "weight": 0.12,
                "provenance": "heuristic",
                "children": {
                    "total": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "patents_count", "weight": 1/6, "alts": [alt("IP.PAT.RESD", "exact", +1)]},
                            {"name": "rd_spend_usd", "weight": 1/6, "alts": [alt("DERIVED.ABS.RD_SPEND_USD", "exact", +1)]},
                            {"name": "researchers_total", "weight": 1/6, "alts": [alt("DERIVED.ABS.RESEARCHERS", "exact", +1)]},
                            {"name": "royalties_received", "weight": 1/6, "alts": [alt("BX.GSR.ROYL.CD", "proxy_concept", +1)]},
                            {"name": "vc_funding", "weight": 1/6, "alts": [alt("VC.TOTAL_FUNDING", "exact", +1, status="GAP", reason="VC funding data paywalled")]},
                            {"name": "cset_ai", "weight": 1/6, "alts": [alt("CSET.AI_OUTPUT", "exact", +1, status="GAP", reason="CSET skipped per D4")]},
                        ]
                    },
                    "per_capita": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "rd_spend_gdp", "weight": 0.20, "alts": [alt("GB.XPD.RSDV.GD.ZS", "exact", +1)]},
                            {"name": "researchers_per_capita", "weight": 0.20, "alts": [alt("SP.POP.SCIE.RD.P6", "exact", +1)]},
                            {"name": "patents_per_capita", "weight": 0.20, "alts": [alt("DERIVED.PER_CAPITA.IP.PAT.RESD", "exact", +1)]},
                            {"name": "wipo_gii", "weight": 0.20, "alts": [alt("WIPO.GII_SCORE", "exact", +1, status="GAP", reason="WIPO GII pending")]},
                            {"name": "bloomberg_innovation", "weight": 0.20, "alts": [alt("BLOOMBERG.INNOVATION_INDEX", "exact", +1, status="GAP", reason="Bloomberg paywalled")]},
                        ]
                    }
                }
            },
            "trade": {
                "weight": 0.10,
                "provenance": "heuristic",
                "children": {
                    "total": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "exports_usd", "weight": 0.50, "alts": [alt("BX.GSR.GNFS.CD", "exact", +1), alt("NE.EXP.GNFS.CD", "substitute_source", +1)]},
                            {"name": "merch_exports_usd", "weight": 0.50, "alts": [alt("TX.VAL.MRCH.CD.WT", "exact", +1)]},
                        ]
                    },
                    "per_capita": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "exports_pc", "weight": 1.0, "alts": [alt("DERIVED.PER_CAPITA.BX.GSR.GNFS.CD", "exact", +1)]},
                        ]
                    }
                }
            },
            "education": {
                "weight": 0.10,
                "provenance": "heuristic",
                "children": {
                    "total": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "sec_enrolment", "weight": 1/3, "alts": [alt("SE.SEC.ENRL", "exact", +1)]},
                            {"name": "prm_enrolment", "weight": 1/3, "alts": [alt("SE.PRM.ENRL", "exact", +1)]},
                            {"name": "arwu_top500", "weight": 1/3, "alts": [alt("ARWU.TOP500_COUNT", "exact", +1)]},
                        ]
                    },
                    "per_capita": {
                        "weight": 0.50,
                        "provenance": "heuristic",
                        "components": [
                            {"name": "pisa", "weight": 1/6, "alts": PISA_MEAN},
                            {"name": "years_schooling", "weight": 1/6, "alts": [alt("BAR.SCHL.25UP", "exact", +1), alt("HD.HCI.EYRS", "proxy_concept", +1)]},
                            {"name": "ter_enrolment_ratio", "weight": 1/6, "alts": [alt("SE.TER.ENRR", "exact", +1)]},
                            {"name": "sec_attainment_ratio", "weight": 1/6, "alts": [alt("SE.SEC.CUAT.UP.ZS", "exact", +1)]},
                            {"name": "edu_spend_gdp", "weight": 1/6, "alts": [alt("SE.XPD.TOTL.GD.ZS", "exact", +1)]},
                            {"name": "arwu_top100_pc", "weight": 1/6, "alts": [alt("DERIVED.PER_CAPITA.ARWU.TOP100_COUNT", "exact", +1)]},
                        ]
                    }
                }
            },
            "competitiveness": {
                "weight": 0.08,
                "provenance": "heuristic",
                "reference": "@productivity_gauge.what_you_pay_vs_what_you_get"
            }
        }
    },
    "additional_measures": {
        "weight": 0.20,
        "provenance": "heuristic",
        "children": {
            "infrastructure_and_investment": {
                "weight": 0.20,
                "provenance": "heuristic",
                "components": [
                    {"name": "gross_capital_formation_usd", "weight": 1/12, "alts": [alt("NE.GDI.FTOT.CD", "exact", +1), alt("NE.GDI.TOTL.CD", "proxy_concept", +1)]},
                    {"name": "investment_pct_gdp", "weight": 1/12, "alts": [alt("NE.GDI.FTOT.ZS", "exact", +1), alt("IMF.NID_NGDP", "proxy_concept", +1)]},
                    {"name": "savings_pct_gdp", "weight": 1/12, "alts": [alt("NY.GNS.ICTR.ZS", "exact", +1), alt("IMF.NGSD_NGDP", "substitute_source", +1)]},
                    {"name": "logistics_perf", "weight": 1/12, "alts": [alt("LP.LPI.OVRL.XQ", "exact", +1)]},
                    {"name": "infra_quality", "weight": 1/12, "alts": [alt("INFRA.QUALITY_INDEX", "exact", +1, status="GAP", reason="Infra quality paywalled")]},
                    {"name": "electricity_gen", "weight": 1/12, "alts": [alt("OWID.ELECTRICITY_GEN_TWH", "exact", +1), alt("EIA.ELECTRICITY.GENERATION.BKWH", "substitute_source", +1)]},
                    {"name": "air_passengers", "weight": 1/12, "alts": [alt("IS.AIR.PSGR", "exact", +1)]},
                    {"name": "air_freight", "weight": 1/12, "alts": [alt("IS.AIR.GOOD.MT.K1", "exact", +1)]},
                    {"name": "rail_lines", "weight": 1/12, "alts": [alt("IS.RRS.TOTL.KM", "exact", +1)]},
                    {"name": "rail_freight", "weight": 1/12, "alts": [alt("IS.RRS.GOOD.MT.K6", "exact", +1)]},
                    {"name": "port_connectivity", "weight": 1/12, "alts": [alt("IS.SHP.GCNW.XQ", "proxy_concept", +1)]},
                    {"name": "internet_users", "weight": 1/12, "alts": [alt("IT.NET.USER.ZS", "exact", +1)]},
                ]
            },
            "resource_allocation_efficiency": {
                "weight": 0.20,
                "provenance": "heuristic",
                "components": [
                    {"name": "unemployment_rate", "weight": 0.20, "alts": [alt("ILO.UNEMP_RATE_MODELLED", "exact", -1), alt("SL.UEM.TOTL.ZS", "substitute_source", -1), alt("IMF.LUR", "substitute_source", -1)]},
                    {"name": "longterm_unemp_share", "weight": 0.20, "alts": [alt("ILO.UNEMP_LONGTERM_SHARE", "exact", -1)]},
                    {"name": "debt_growth_minus_income", "weight": 0.20, "alts": [alt("DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH", "exact", -1)]},
                    {"name": "ease_getting_loans", "weight": 0.20, "alts": [alt("IC.CRED.ACC.LGL.RGHT.012.XD.DB1519.DFRN", "proxy_concept", +1)]},
                    {"name": "labor_rigidity_epl", "weight": 0.20, "alts": [alt("OECD.EPL", "exact", -1, status="GAP", reason="OECD EPL pending")]},
                ]
            },
            "geology": {
                "weight": 0.20,
                "provenance": "heuristic",
                "components": [
                    {"name": "land_area", "weight": 1/13, "alts": [alt("AG.LND.TOTL.K2", "exact", +1)]},
                    {"name": "energy_production", "weight": 1/13, "alts": [alt("EIA.ENERGY.PRODUCTION_QBTU", "exact", +1), alt("OWID.PRIMARY_ENERGY_TWH", "proxy_concept", +1)]},
                    {"name": "oil_production", "weight": 1/13, "alts": [alt("OWID.OIL_PRODUCTION_TWH", "exact", +1)]},
                    {"name": "gas_production", "weight": 1/13, "alts": [alt("OWID.GAS_PRODUCTION_TWH", "exact", +1)]},
                    {"name": "coal_production", "weight": 1/13, "alts": [alt("OWID.COAL_PRODUCTION_TWH", "exact", +1)]},
                    {"name": "food_production", "weight": 1/13, "alts": [alt("AG.PRD.CREL.MT", "exact", +1), alt("AG.PRD.FOOD.XD", "proxy_concept", +1)]},
                    {"name": "metals_production", "weight": 1/13, "alts": [alt("USGS.METALS_PRODUCTION", "exact", +1, status="GAP", reason="USGS metals pending")]},
                    {"name": "natural_resource_rents", "weight": 1/13, "alts": [alt("NY.GDP.TOTL.RT.ZS", "exact", +1)]},
                    {"name": "energy_self_suff", "weight": 1/13, "alts": [alt("EIA.ENERGY.SELF_SUFF", "exact", +1), alt("EG.IMP.CONS.ZS", "proxy_concept", -1)]},
                    {"name": "net_exports_fuel", "weight": 1/13, "alts": [alt("DERIVED.NET_EXPORTS_FUEL_PCT_GDP", "exact", +1)]},
                    {"name": "net_exports_agri", "weight": 1/13, "alts": [alt("DERIVED.NET_EXPORTS_AGRI_PCT_GDP", "exact", +1)]},
                    {"name": "net_exports_food", "weight": 1/13, "alts": [alt("DERIVED.NET_EXPORTS_FOOD_PCT_GDP", "exact", +1)]},
                    {"name": "freshwater_resources", "weight": 1/13, "alts": [alt("ER.H2O.INTR.K3", "exact", +1), alt("ER.H2O.INTR.PC", "proxy_concept", +1)]},
                ]
            },
            "character_determination_civility": {
                "weight": 0.15,
                "provenance": "heuristic",
                "components": [
                    {"name": "hours_worked", "weight": 1/6, "alts": HOURS_WORKED_WEEKLY},
                    {"name": "male_lfpr", "weight": 1/6, "alts": [alt("SL.TLF.CACT.MA.ZS", "exact", +1), alt("ILO.LFPR_MODELLED_MALE", "substitute_source", +1)]},
                    {"name": "gov_transfers", "weight": 1/6, "alts": [alt("GC.XPN.TRFT.ZS", "exact", -1)]},
                    {"name": "effective_retirement_age", "weight": 1/6, "alts": [alt("OECD.EFFECTIVE_RETIREMENT_AGE", "exact", +1, status="GAP", reason="OECD pension age pending")]},
                    {"name": "hard_work_success_survey", "weight": 1/6, "alts": [alt("WVS.HARD_WORK_LEADS_TO_SUCCESS", "exact", +1, status="GAP", reason="WVS pending")]},
                    {"name": "success_important_survey", "weight": 1/6, "alts": [alt("WVS.SUCCESS_IMPORTANT", "exact", +1, status="GAP", reason="WVS pending")]},
                ]
            },
            "governance_rule_of_law": {
                "weight": 0.15,
                "provenance": "heuristic",
                "components": [
                    {"name": "rule_of_law_wgi", "weight": 1/6, "alts": [alt("RL.EST", "substitute_source", +1)]},
                    {"name": "corruption_wgi", "weight": 1/6, "alts": [alt("CC.EST", "substitute_source", +1)]},
                    {"name": "gov_effectiveness_wgi", "weight": 1/6, "alts": [alt("GE.EST", "proxy_concept", +1)]},
                    {"name": "regulatory_quality_wgi", "weight": 1/6, "alts": [alt("RQ.EST", "proxy_concept", +1)]},
                    {"name": "rule_of_law_vdem", "weight": 1/6, "alts": [alt("VDEM.v2x_rule", "substitute_source", +1)]},
                    {"name": "corruption_vdem", "weight": 1/6, "alts": [alt("VDEM.v2x_corr", "proxy_concept", -1)]},
                ]
            },
            "acts_of_nature": {
                "weight": 0.10,
                "provenance": "heuristic",
                "components": [
                    {"name": "climate_vulnerability", "weight": 1/3, "alts": [alt("NDGAIN.VULNERABILITY", "proxy_concept", -1)]},
                    {"name": "disaster_readiness", "weight": 1/3, "alts": [alt("NDGAIN.READINESS", "substitute_source", +1)]},
                    {"name": "covid_excess_mortality", "weight": 1/3, "alts": [alt("COVID.EXCESS_MORTALITY_PSCORE", "substitute_source", -1)]},
                ]
            }
        }
    },
    "internal_order_cycle": {
        "weight": 1.0,
        "provenance": "heuristic",
        "children": {
            "wealth_values_gap": {
                "weight": 0.50,
                "provenance": "heuristic",
                "components": [
                    {"name": "gini", "weight": 1/7, "alts": [alt("SI.POV.GINI", "exact", -1), alt("PIP.GINI", "substitute_source", -1)]},
                    {"name": "top1_income", "weight": 1/7, "alts": [alt("WID.TOP1_INCOME", "exact", -1)]},
                    {"name": "top10_income", "weight": 1/7, "alts": [alt("WID.TOP10_INCOME", "exact", -1), alt("SI.DST.10TH.10", "proxy_concept", -1), alt("PIP.DECILE10", "proxy_concept", -1)]},
                    {"name": "top1_wealth", "weight": 1/7, "alts": [alt("WID.TOP1_WEALTH", "exact", -1)]},
                    {"name": "top10_wealth", "weight": 1/7, "alts": [alt("WID.TOP10_WEALTH", "exact", -1)]},
                    {"name": "equal_distribution_resources", "weight": 1/7, "alts": [alt("VDEM.v2xeg_eqdr", "proxy_concept", +1)]},
                    {"name": "polarization", "weight": 1/7, "alts": [alt("VDEM.v2cacamps", "proxy_concept", -1), alt("VDEM.v2smpolsoc", "proxy_concept", -1)]},
                ]
            },
            "internal_conflict": {
                "weight": 0.50,
                "provenance": "heuristic",
                "components": [
                    {"name": "protests_vdem", "weight": 1/7, "alts": [alt("VDEM.v2cagenmob", "proxy_concept", -1), alt("VDEM.v2cademmob", "proxy_concept", -1)]},
                    {"name": "physical_violence_vdem", "weight": 1/7, "alts": [alt("VDEM.v2x_clphy", "proxy_concept", +1)]},
                    {"name": "political_violence_wgi", "weight": 1/7, "alts": [alt("PV.EST", "proxy_concept", +1)]},
                    {"name": "internal_conflict_deaths", "weight": 1/7, "alts": [alt("UCDP.INTERNAL_CONFLICT_DEATHS", "substitute_source", -1)]},
                    {"name": "strikes_days_lost", "weight": 1/7, "alts": [alt("ILO.STRIKES_DAYS_NOT_WORKED", "exact", -1)]},
                    {"name": "strikes_count", "weight": 1/7, "alts": [alt("ILO.STRIKES_COUNT", "exact", -1)]},
                    {"name": "conflict_media_discontent", "weight": 1/7, "alts": [
                        alt("GDELT.PROTEST_INTENSITY", "exact", -1, status="GAP", reason="GDELT pending"),
                        alt("GDELT.GOVT_DISAPPROVAL", "exact", -1, status="GAP", reason="GDELT pending"),
                        alt("GDELT.DISCONTENT_TERMS", "exact", -1, status="GAP", reason="GDELT pending"),
                        alt("CLINE.CONFLICT_PER_CAPITA", "exact", -1, status="GAP", reason="Cline pending"),
                    ]},
                ]
            }
        }
    },
    "economic_financial_cycle": {
        "weight": 1.0,
        "provenance": "heuristic",
        "children": {
            "debt_burden": {
                "weight": 0.50,
                "provenance": "heuristic",
                "reference": "@indebtedness_gauge"
            },
            "expected_growth": {
                "weight": 0.50,
                "provenance": "heuristic",
                "reference": "@future_growth"
            }
        }
    },
    "external_order_cycle": {
        "weight": 1.0,
        "provenance": "heuristic",
        "components": [
            {"name": "tariff_rate", "weight": 0.20, "alts": [alt("TM.TAX.MRCH.WM.AR.ZS", "exact", -1)]},
            {"name": "trade_concentration", "weight": 0.20, "alts": [alt("DOTS.BILATERAL_TRADE_CONCENTRATION", "exact", -1, status="GAP", reason="Pairwise major-power gauge; no country-level free source defined")]},
            {"name": "un_voting_distance", "weight": 0.20, "alts": [alt("UN.VOTING_DISTANCE", "exact", -1, status="GAP", reason="Pairwise major-power gauge; no country-level free source defined")]},
            {"name": "sanctions_exposure", "weight": 0.20, "alts": [alt("SANCTIONS.GSDB", "exact", -1, status="GAP", reason="Pairwise major-power gauge; no country-level free source defined")]},
            {"name": "interstate_conflict", "weight": 0.20, "alts": [alt("UCDP.INTERSTATE", "exact", -1, status="GAP", reason="Pairwise major-power gauge; no country-level free source defined")]},
        ]
    }
}

# Tree Utilities & Weight Verification (FRD §5, Appendix C)
def is_leaf(node):
    return "components" in node or "reference" in node

def iter_leaves(tree_dict, path=(), eff_weight=1.0):
    for name, node in tree_dict.items():
        curr_path = path + (name,)
        w = eff_weight * node.get("weight", 1.0)
        if "reference" in node:
            yield "/".join(curr_path), {"reference": node["reference"]}, w
        elif "components" in node:
            yield "/".join(curr_path), node["components"], w
        elif "children" in node:
            yield from iter_leaves(node["children"], curr_path, w)

def check_tree_weights(tree_dict):
    discrepancies = []
    def walk(node, path_str):
        if "children" in node:
            children = node["children"]
            total_w = sum(ch.get("weight", 0.0) for ch in children.values())
            if abs(total_w - 1.0) > 1e-3:
                discrepancies.append((path_str, round(total_w, 4)))
            for ch_name, ch_node in children.items():
                walk(ch_node, f"{path_str}/{ch_name}")
        elif "components" in node:
            comps = node["components"]
            total_comp_w = sum(c.get("weight", 0.0) for c in comps)
            if abs(total_comp_w - 1.0) > 1e-3:
                discrepancies.append((f"{path_str} (components)", round(total_comp_w, 4)))
    for name, root in tree_dict.items():
        walk(root, name)
    return discrepancies

# Assert tree weights and verify hours worked arithmetic
disc = check_tree_weights(RAY_DALIO_GAUGE_TREES)
print("Weight discrepancies (known exceptions):", disc)
for p, s in disc:
    if "investing" in p:
        assert abs(s - 1.0057) < 1e-3, f"Expected investing sum ~1.0057, got {s}"
    elif "strength_determinants" in p:
        assert abs(s - 0.94) < 1e-3, f"Expected key_eight sum ~0.94, got {s}"
    else:
        raise AssertionError(f"Unexpected weight sum {s} at {p}")

# Hours worked effective weight in productivity gauge (Appendix C: 15.45% ± 0.1%)
prod_gauge = {"productivity_gauge": RAY_DALIO_GAUGE_TREES["productivity_gauge"]}
hours_eff_weight = sum(
    eff_w * comp.get("weight", 1.0)
    for leaf_path, comps, eff_w in iter_leaves(prod_gauge)
    if isinstance(comps, list)
    for comp in comps
    if "ILO.HOURS_WEEKLY_MODELLED_MALE" in [a.code for a in comp.get("alts", [])]
)
print(f"Hours-worked effective weight in productivity gauge: {hours_eff_weight*100:.3f}%")
assert 0.1535 <= hours_eff_weight <= 0.1555, f"Hours weight {hours_eff_weight*100:.3f}% not in 15.45% ± 0.1%"

# --- Retention Filtering (FRD P-6) ---
# After tree is final, compute KEEP = codes in tree ∪ derived inputs ∪ Appendix §13 list ∪ registered stand-ins ∪ ND-GAIN
keep_codes = set()
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for comp in comps:
            for alt_obj in comp.get("alts", []):
                keep_codes.add(alt_obj.code)

ndgain_codes = set(df_master[df_master["indicator_code"].str.startswith("NDGAIN.")]["indicator_code"].unique())
keep_codes |= ndgain_codes

derived_inputs = {
    "SP.POP.1564.TO", "PROJ.SP.POP.1564.TO", "SP.POP.DPND", "PROJ.SP.POP.DPND",
    "NY.GDP.MKTP.KD.ZG", "FP.CPI.TOTL.ZG", "NE.GDI.FTOT.ZS", "UNHCR.DISPLACED_TOTAL",
    "SP.POP.TOTL", "PWT.AVH", "LO.PISA.MAT", "LO.PISA.REA", "LO.PISA.SCI",
    "GB.XPD.RSDV.GD.ZS", "NY.GDP.MKTP.CD", "SP.POP.SCIE.RD.P6", "MS.MIL.XPND.CD",
    "MS.MIL.TOTL.P1", "IP.PAT.RESD", "BX.GSR.GNFS.CD", "ARWU.TOP100_COUNT",
    "TX.VAL.MRCH.CD.WT", "TM.VAL.MRCH.CD.WT", "TX.VAL.FUEL.ZS.UN", "TM.VAL.FUEL.ZS.UN",
    "TX.VAL.AGRI.ZS.UN", "TM.VAL.AGRI.ZS.UN", "TX.VAL.FOOD.ZS.UN", "TM.VAL.FOOD.ZS.UN",
    "TX.VAL.MMTL.ZS.UN", "TM.VAL.MMTL.ZS.UN", "BIS.TC.N.PCT_GDP", "FS.AST.PRVT.GD.ZS",
    "IMF.GGXWDG_NGDP"
}
keep_codes |= derived_inputs

df_master = df_master[df_master["indicator_code"].isin(keep_codes)].reset_index(drop=True)
print(f"Master table filtered to KEEP: {len(df_master)} observations across {df_master['country_code'].nunique()} countries and {df_master['indicator_code'].nunique()} series.")


# 6. Minimal Scoring Engine (§6)
Computes annual gauge scores from `YEAR_FLOOR = 1950` to the last actual year.
Steps:
1. Wide country × year matrices.
2. Standardisation: percentile rank within (code, year) across countries; ties averaged, scaled 0–100.
   Valid only if cross-section count ≥ `MIN_CROSS_SECTION = 30`.
3. Coalescing (D16): first available alternative supplies value. Proxy fallbacks excluded unless `ALLOW_PROXY_FALLBACK=True`.
4. Bottom-up renormalisation (D20): missing components/children are renormalised over present children.
5. Reference resolution: `@dotted.path` references evaluated before consuming nodes.
6. Exports parquet files to `data/dalitest/out/`.


In [ ]:
# 6. Minimal Scoring Engine (§6)
def run_gauge_engine(df, trees, year_floor=1950, allow_proxy_fallback=False, min_cross_section=30):
    non_proj = df[~df["is_projection"].astype(bool)].copy() if (not df.empty and "is_projection" in df.columns) else df.copy()
    if "year" in non_proj.columns and not non_proj.empty:
        last_actual_year = int(non_proj["year"].max())
        years = sorted(y for y in range(year_floor, last_actual_year + 1) if y in non_proj["year"].values)
    else:
        last_actual_year = 2024
        years = [2024]
    if not years:
        years = [2024]
    countries = UNIVERSE
    
    # 1. Wide matrices (indicator -> DataFrame[index=years, columns=countries])
    wide = {}
    if "indicator_code" in non_proj.columns and not non_proj.empty:
        for code, g in non_proj.groupby("indicator_code"):
            w = g.pivot_table(index="year", columns="country_code", values="value", aggfunc="first")
            w = w.reindex(index=years, columns=countries)
            wide[code] = w

    # Map code -> alt directions and tiers
    code_directions = {}
    code_tiers = {}
    for leaf_path, comps, _ in iter_leaves(trees):
        if isinstance(comps, list):
            for c in comps:
                for a in c.get("alts", []):
                    code_directions[a.code] = a.direction
                    code_tiers[a.code] = a.tier

    # 2. Vectorized Ranking per (indicator, year)
    ranked_mats = {}
    rank_support_rows = []
    empty_mat = pd.DataFrame(np.nan, index=years, columns=countries)
    
    for code, w in wide.items():
        dir_val = code_directions.get(code, +1)
        directed = w * dir_val
        counts = directed.count(axis=1)
        r_df = empty_mat.copy()
        for y in years:
            n = int(counts.get(y, 0))
            rank_support_rows.append({"indicator_code": code, "year": y, "n_ranked": n})
            if n >= min_cross_section:
                row = directed.loc[y].dropna()
                rk = row.rank(method="average", ascending=True)
                pct = (rk - 1.0) / (n - 1.0) * 100.0
                r_df.loc[y, pct.index] = pct
        ranked_mats[code] = r_df

    df_rank_support = pd.DataFrame(rank_support_rows)

    # 3. Component coalescing and Leaf scoring
    comp_score_records = []
    leaf_scores = {}
    leaf_w_pres = {}
    tree_codes_in_master = set(df["indicator_code"].unique()) if ("indicator_code" in df.columns and not df.empty) else set()

    for leaf_path, comps, _ in iter_leaves(trees):
        if not isinstance(comps, list):
            continue
        
        leaf_comp_scores = []
        leaf_comp_weights = []
        
        for comp in comps:
            comp_name = comp["name"]
            c_weight = comp.get("weight", 1.0)
            alts = comp.get("alts", [])
            active_alts = [a for a in alts if a.code in tree_codes_in_master]
            
            c_score = empty_mat.copy()
            c_alt_used = pd.DataFrame(-1, index=years, columns=countries)
            
            for a_idx, a in enumerate(active_alts):
                is_primary = (a_idx == 0)
                eligible = is_primary or (a.tier in ("exact", "substitute_source")) or allow_proxy_fallback
                if not eligible:
                    continue
                
                r_mat = ranked_mats.get(a.code)
                if r_mat is not None:
                    fill_mask = c_score.isna() & r_mat.notna()
                    c_score[fill_mask] = r_mat[fill_mask]
                    c_alt_used[fill_mask] = a_idx
            
            leaf_comp_scores.append(c_score)
            leaf_comp_weights.append(c_weight)
            
            # Stack into long format
            s_stacked = c_score.stack(future_stack=True).reset_index()
            s_stacked.columns = ["year", "country_code", "score"]
            alt_stacked = c_alt_used.stack(future_stack=True).values
            s_stacked["alt_used"] = alt_stacked
            s_stacked["observed"] = s_stacked["score"].notna()
            s_stacked["leaf_path"] = leaf_path
            s_stacked["component_name"] = comp_name
            comp_score_records.append(s_stacked)

        # Leaf score via weighted mean of present components (FRD D20)
        total_nom_weight = sum(leaf_comp_weights)
        num = empty_mat.copy().fillna(0.0)
        den = empty_mat.copy().fillna(0.0)
        
        for w, s in zip(leaf_comp_weights, leaf_comp_scores):
            present = s.notna()
            num += w * s.fillna(0.0)
            den += w * present.astype(float)
        
        leaf_s = num / den.replace(0.0, np.nan)
        leaf_wp = den / total_nom_weight
        leaf_scores[leaf_path] = leaf_s
        leaf_w_pres[leaf_path] = leaf_wp

    comp_cols = ["country_code", "year", "leaf_path", "component_name", "score", "alt_used", "observed"]
    df_comp_scores = pd.concat(comp_score_records, ignore_index=True)[comp_cols] if comp_score_records else pd.DataFrame(columns=comp_cols)

    # 4. Resolve References and Bottom-up Node Scores
    node_scores = {}
    node_w_pres = {}

    def resolve_ref(ref_str):
        return "/".join(ref_str.lstrip("@").split("."))

    def eval_node_mat(node, path_str):
        if "reference" in node:
            ref_path = resolve_ref(node["reference"])
            return node_scores[ref_path], node_w_pres[ref_path]
        elif "components" in node:
            return leaf_scores[path_str], leaf_w_pres[path_str]
        elif "children" in node:
            ch_scores = []
            ch_weights = []
            ch_wps = []
            for ch_name, ch_node in node["children"].items():
                ch_path = f"{path_str}/{ch_name}"
                s, wp = eval_node_mat(ch_node, ch_path)
                w = ch_node.get("weight", 1.0)
                ch_scores.append(s)
                ch_weights.append(w)
                ch_wps.append(wp)
            
            total_nom_w = sum(ch_weights)
            num = empty_mat.copy().fillna(0.0)
            den = empty_mat.copy().fillna(0.0)
            wp_num = empty_mat.copy().fillna(0.0)
            
            for w, s, wp in zip(ch_weights, ch_scores, ch_wps):
                present = s.notna()
                num += w * s.fillna(0.0)
                den += w * present.astype(float)
                wp_num += w * wp.fillna(0.0)
            
            n_score = num / den.replace(0.0, np.nan)
            n_wp = wp_num / total_nom_w
            return n_score, n_wp

    eval_order = [
        "health_index", "happiness_index", "productivity_gauge", "indebtedness_gauge",
        "future_growth", "strength_determinants", "additional_measures",
        "internal_order_cycle", "economic_financial_cycle", "external_order_cycle"
    ]
    node_score_records = []
    
    for tree_name in eval_order:
        tree_root = trees[tree_name]
        def walk_eval(node, path_str):
            if "children" in node:
                for ch_name, ch_node in node["children"].items():
                    walk_eval(ch_node, f"{path_str}/{ch_name}")
            s, wp = eval_node_mat(node, path_str)
            node_scores[path_str] = s
            node_w_pres[path_str] = wp
            
            stacked_s = s.stack(future_stack=True).reset_index()
            stacked_s.columns = ["year", "country_code", "score"]
            stacked_wp = wp.stack(future_stack=True).values
            stacked_s["weight_present"] = stacked_wp
            stacked_s["node_path"] = path_str
            node_score_records.append(stacked_s)
            
        walk_eval(tree_root, tree_name)

    node_cols = ["country_code", "year", "node_path", "score", "weight_present"]
    df_node_scores = pd.concat(node_score_records, ignore_index=True)[node_cols] if node_score_records else pd.DataFrame(columns=node_cols)

    # Save to parquet (FRD §6.2)
    df_comp_scores.to_parquet(OUT_DIR / "component_scores.parquet", index=False)
    df_node_scores.to_parquet(OUT_DIR / "node_scores.parquet", index=False)
    df_rank_support.to_parquet(OUT_DIR / "rank_support.parquet", index=False)
    print("Engine complete! Outputs written to data/dalitest/out/.")

    return df_comp_scores, df_node_scores, df_rank_support

# Execute Engine
df_comp_scores, df_node_scores, df_rank_support = run_gauge_engine(
    df_master, RAY_DALIO_GAUGE_TREES, year_floor=YEAR_FLOOR,
    allow_proxy_fallback=ALLOW_PROXY_FALLBACK, min_cross_section=MIN_CROSS_SECTION
)


# 7a. QA Reports & PDF Ledger (§7.1)
Generates all diagnostics specified in FRD §7.1 and exports to `data/dalitest/out/`:
- `df_health`: indicator-level health with PPP GDP weights, tier, stale flag, first valid year.
- `pdf_ledger.csv`: complete ledger of 64 Appendix items + §11 non-list inputs.
- `df_registry.csv`: series registry with consumer status.
- `alt_coverage.csv`: alternative coverage by leaf and component.
- `effective_weights.csv`: effective weights and tier/provenance distribution.
- `switch_report.csv`: alternative switch counts across countries.
- `agreement_report.csv`: empirical Spearman correlation between alternative pairs.
- `proxy_candidates.csv`: coverage impact of excluded proxies.
- `direction_map.csv`: complete direction map with ambiguous flags.


In [ ]:
# 7a. QA Reports & PDF Ledger

# Helper to index df_master by code without repeatedly filtering
wide_map = {
    code: g.set_index(["year", "country_code"])["value"] 
    for code, g in df_master.groupby("indicator_code")
} if ("indicator_code" in df_master.columns and not df_master.empty) else {}

# Build leaf & tier mappings
code_leaf_map = defaultdict(list)
code_tier_map = {}
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for c in comps:
            for a in c.get("alts", []):
                code_leaf_map[a.code].append(leaf_path)
                code_tier_map[a.code] = a.tier

# 1. df_health (FRD §7.1 - Restored & Enriched)
health_rows = []

if "indicator_code" in df_master.columns and not df_master.empty:
    for code, g in df_master.groupby("indicator_code"):
        # Separate actuals from projections
        actuals = g[~g["is_projection"]]
        has_proj = bool(g["is_projection"].any())
        
        # If no actuals, evaluate on full group but flag accordingly
        eval_df = actuals if not actuals.empty else g
        
        all_rep = set(eval_df["country_code"].unique())
        target_rep = all_rep & TARGET_SET
        recent_rep = set(eval_df[eval_df["year"] >= STALE_BEFORE]["country_code"].unique())
        target_recent_rep = recent_rep & TARGET_SET
        
        # Coverage calculations
        raw_cov_pct = (len(all_rep) / len(UNIVERSE)) * 100.0
        target_cov_pct = (len(target_rep) / len(TARGET_SET) * 100.0) if TARGET_SET else 0.0
        w_cov_pct = OBS_WEIGHTS[list(all_rep)].sum() * 100.0
        
        recent_cov_pct = (len(recent_rep) / len(UNIVERSE)) * 100.0
        w_recent_cov_pct = OBS_WEIGHTS[list(recent_rep)].sum() * 100.0
        
        # Latest actual year and freshness
        latest_actual_y = int(actuals["year"].max()) if not actuals.empty else int(g["year"].max())
        stale = latest_actual_y < STALE_BEFORE
        
        # Cross-sectional depth (based on actuals)
        counts_by_year = actuals.groupby("year")["country_code"].nunique() if not actuals.empty else g.groupby("year")["country_code"].nunique()
        valid_years = counts_by_year[counts_by_year >= MIN_CROSS_SECTION].index
        first_valid_y = int(valid_years.min()) if len(valid_years) > 0 else np.nan
        
        # Degeneracy: check std == 0, null, or fewer than 5 unique values
        std_val = eval_df["value"].std()
        distinct_vals = eval_df["value"].nunique()
        degenerate = bool((std_val == 0.0 or pd.isna(std_val)) and distinct_vals < 5)
        
        leaves = code_leaf_map.get(code, [])
        gauge = leaves[0].split("/")[0] if leaves else "unassigned"
        tier = code_tier_map.get(code, "exact")
        
        health_rows.append({
            "gauge": gauge,
            "indicator": code,
            "source": g["source"].iloc[0] if "source" in g.columns else "UNKNOWN",
            "reporting_countries": len(all_rep),
            "target_reporting": len(target_rep),
            "global_cov_%": round(raw_cov_pct, 1),
            "target_cov_%": round(target_cov_pct, 1),
            "weighted_cov_%": round(w_cov_pct, 1),
            f"weighted_recent_% (>={STALE_BEFORE})": round(w_recent_cov_pct, 1),
            "latest_year": latest_actual_y,
            "stale": stale,
            "first_valid_year": first_valid_y,
            "degenerate": degenerate,
            "has_projection": has_proj,
            "tier": tier,
            "status": "wired" if leaves else "pdf_listed_unwired"
        })

df_health = pd.DataFrame(health_rows).sort_values(
    by=["gauge", "weighted_cov_%", f"weighted_recent_% (>={STALE_BEFORE})"], 
    ascending=[True, False, False]
) if health_rows else pd.DataFrame()
df_health.to_csv(OUT_DIR / "df_health.csv", index=False)

# 2. PDF Ledger (FRD §13, §11)
LEDGER_DATA = [
    (1, "Access to essential health services", "Social Progress Imperative", ["SH.UHC.SRVS.CV.XD"], "substitute", "substitute", None),
    (2, "Aerospace and defense sector returns", "Thomson Reuters", [], "exact", "gap", "Paywalled"),
    (3, "Agriculture, forestry, fishing value added", "WDI", ["NV.AGR.TOTL.ZS", "NV.AGR.TOTL.CD"], "exact", "live", None),
    (4, "AI: new companies, patents, citations", "CSET", ["CSET.AI_OUTPUT"], "exact", "skipped", "CSET skipped per D4"),
    (5, "Banking claims by currency and source country", "BIS", ["BIS.LBS.CROSSBORDER_CLAIMS"], "exact", "gap", "BIS claims bulk file pending"),
    (6, "Bloomberg Innovation Index", "Bloomberg", ["BLOOMBERG.INNOVATION_INDEX"], "exact", "gap", "Paywalled"),
    (7, "Book titles per capita", "CLIO Infra", [], "exact", "gap", "Historical dataset, no current programmatic consumer"),
    (8, "Child mortality rate", "Social Progress Imperative", ["SH.DYN.MORT"], "substitute", "substitute", None),
    (9, "Climate vulnerability and preparedness, subcomponents", "ND-GAIN", ["NDGAIN.VULNERABILITY", "NDGAIN.READINESS"], "proxy", "live", None),
    (10, "CO2 emissions", "CLIO Infra", ["EN.GHG.CO2.MT.CE.AR5", "EN.GHG.CO2.PC.CE.AR5"], "substitute", "live", None),
    (11, "Conflict intensity and count per capita", "Cline Center", ["CLINE.CONFLICT_PER_CAPITA"], "exact", "gap", "Cline Center public API pending"),
    (12, "Construction permits score", "WB Doing Business", ["IC.CNST.PRMT.DFRN.DB1619", "IC.CNST.PRMT.DFRN.DB0615"], "exact", "live", None),
    (13, "Days lost to labor disputes", "CLIO Infra", ["ILO.STRIKES_DAYS_NOT_WORKED"], "substitute", "substitute", None),
    (14, "Discontent, populism, conflict terms in media", "GDELT", ["GDELT.DISCONTENT_TERMS"], "exact", "gap", "GDELT API pending"),
    (15, "Education spending", "WDI", ["SE.XPD.TOTL.GD.ZS"], "exact", "live", None),
    (16, "Educational attainment", "Barro-Lee", ["BAR.SCHL.25UP"], "exact", "live", None),
    (17, "Electricity production", "NBER CHAT", ["OWID.ELECTRICITY_GEN_TWH", "EIA.ELECTRICITY.GENERATION.BKWH"], "substitute", "substitute", None),
    (18, "Energy production and consumption", "EIA", ["EIA.ENERGY.PRODUCTION_QBTU", "EIA.ENERGY.SELF_SUFF"], "exact", "live", None),
    (19, "Enforcing contracts score", "WB Doing Business", ["ENF.CONT.COEN.DB1719.DFRN", "ENF.CONT.COEN.DB0415"], "exact", "live", None),
    (20, "GINI coefficient", "WDI", ["SI.POV.GINI", "PIP.GINI"], "exact", "live", None),
    (21, "Google searches for political conflict terms", "Google Trends", [], "exact", "skipped", "Google Trends skipped per D4"),
    (22, "Google searches indicating discontent", "Google Trends", [], "exact", "skipped", "Google Trends skipped per D4"),
    (23, "Government disapproval, verbal opposition in news", "GDELT", ["GDELT.GOVT_DISAPPROVAL"], "exact", "gap", "GDELT API pending"),
    (24, "Gross expenditure on R&D", "WIPO GII", ["GB.XPD.RSDV.GD.ZS"], "substitute", "substitute", None),
    (25, "Historical aviation passengers and cargo", "NBER CHAT", ["IS.AIR.PSGR", "IS.AIR.GOOD.MT.K1"], "substitute", "substitute", None),
    (26, "Historical technology adoption", "NBER CHAT", [], "exact", "gap", "Historical dataset, no 2024 gauge consumer"),
    (27, "Homicide rate", "Social Progress Imperative", ["VC.IHR.PSRC.P5"], "substitute", "substitute", None),
    (28, "Hours worked per week", "World Inequality Database", ["ILO.HOURS_WEEKLY_MODELLED_MALE", "ILO.HOURS_WEEKLY_MODELLED", "DERIVED.PWT_HOURS_WEEKLY"], "exact", "live", None),
    (29, "Human Capital Index", "World Bank", ["HD.HCI.OVRL", "HD.HCI.EYRS", "HD.HCI.HLOS"], "exact", "live", None),
    (30, "Income distributions", "World Bank", ["SI.DST.FRST.20", "SI.DST.05TH.20", "SI.DST.10TH.10", "PIP.DECILE1", "PIP.DECILE10"], "exact", "live", None),
    (31, "Internal armed conflict", "CLIO Infra", ["UCDP.INTERNAL_CONFLICT_DEATHS"], "substitute", "substitute", None),
    (32, "Internet users", "NBER CHAT", ["IT.NET.USER.ZS"], "substitute", "substitute", None),
    (33, "Male labor force participation", "World Bank", ["SL.TLF.CACT.MA.ZS"], "exact", "live", None),
    (34, "Metals and ores production", "CLIO Infra", ["USGS.METALS_PRODUCTION"], "exact", "gap", "USGS metals pending"),
    (35, "Nuclear weapon stockpile", "Federation of American Scientists", ["FAS.NUCLEAR_STOCKPILE"], "exact", "live", None),
    (36, "Number of new patents", "WIPO GII", ["IP.PAT.RESD"], "substitute", "substitute", None),
    (37, "Numeracy", "CLIO Infra", [], "exact", "gap", "Historical dataset, no current consumer"),
    (38, "OTC FX turnover", "BIS", ["BIS.FX_TURNOVER_VALUE"], "exact", "gap", "BIS triennial FX survey programmatic feed pending"),
    (39, "Particulate matter pollution", "Social Progress Imperative", ["EN.ATM.PM25.MC.M3"], "substitute", "substitute", None),
    (40, "Patent applications", "WDI", ["IP.PAT.RESD", "IP.PAT.NRES"], "exact", "live", None),
    (41, "People displaced by conflict and violence", "WDI", ["UNHCR.DISPLACED_PCT_POP"], "substitute", "substitute", None),
    (42, "People using safely managed drinking water", "World Bank", ["SH.H2O.SMDW.ZS"], "exact", "live", None),
    (43, "Prevalence of undernourishment", "World Bank", ["SN.ITK.DEFC.ZS"], "exact", "live", None),
    (44, "Protecting investors value score", "WB Doing Business", ["PROT.MINOR.INV.DFRN.DB1519", "PROT.MINOR.INV.DFRN.DB0614"], "exact", "live", None),
    (45, "Protest intensity index", "GDELT", ["GDELT.PROTEST_INTENSITY"], "exact", "gap", "GDELT API pending"),
    (46, "Public sector corruption and theft", "V-Dem", ["VDEM.v2exthftps", "VDEM.v2excrptps"], "exact", "live", None),
    (47, "Rail lines opened and rail cargo", "NBER CHAT", ["IS.RRS.TOTL.KM", "IS.RRS.GOOD.MT.K6"], "substitute", "substitute", None),
    (48, "Real GDP per worker", "Penn World Table", ["PWT.RGDPO_PER_WORKER"], "exact", "live", None),
    (49, "Real growth rate", "World Bank", ["NY.GDP.MKTP.KD.ZG", "IMF.NGDP_RPCH"], "exact", "live", None),
    (50, "Renewable freshwater resources", "WDI", ["ER.H2O.INTR.K3", "ER.H2O.INTR.PC"], "exact", "live", None),
    (51, "Rigorous and impartial government score", "V-Dem", ["VDEM.v2clrspct"], "exact", "live", None),
    (52, "Royalty and license fees", "WIPO GII", ["BM.GSR.ROYL.CD", "BX.GSR.ROYL.CD"], "substitute", "substitute", None),
    (53, "Share of universities in top 500", "Shanghai Ranking", ["ARWU.TOP100_COUNT", "ARWU.TOP500_COUNT"], "exact", "live", None),
    (54, "Shipping tonnage by sail, steam, motor and total", "NBER CHAT", ["IS.SHP.GCNW.XQ"], "proxy", "proxy", None),
    (55, "Starting a business score", "WB Doing Business", ["IC.REG.DFRN.PC.DFRN"], "exact", "live", None),
    (56, "Suicide mortality rate", "World Bank", ["SH.STA.SUIC.P5"], "exact", "live", None),
    (57, "Tariff rate", "World Bank", ["TM.TAX.MRCH.WM.AR.ZS"], "exact", "live", None),
    (58, "Top 1% and top 10% income shares", "World Inequality Database", ["WID.TOP1_INCOME", "WID.TOP10_INCOME"], "exact", "live", None),
    (59, "Top 1% and top 10% wealth shares", "World Inequality Database", ["WID.TOP1_WEALTH", "WID.TOP10_WEALTH"], "exact", "live", None),
    (60, "Total labor disputes", "CLIO Infra", ["ILO.STRIKES_COUNT"], "substitute", "substitute", None),
    (61, "Transparent laws with predictable enforcement", "V-Dem", ["VDEM.v2cltrnslw"], "exact", "live", None),
    (62, "Universities founded", "CLIO Infra", [], "exact", "gap", "Historical dataset, no current consumer"),
    (63, "Number of new businesses, people creating businesses", "WIPO GII", ["IC.BUS.NREG", "IC.BUS.NDNS.ZS"], "substitute", "substitute", None),
    (64, "Number of new major websites", "WIPO GII", ["WIPO.NEW_MAJOR_WEBSITES"], "exact", "gap", "WIPO web metrics feed pending"),
    ("WVS", "World Values Survey items (8)", "World Values Survey", ["WVS.HAPPY", "WVS.CHILDREN_RESPONSIBILITY", "WVS.GROWTH_IMPORTANT", "WVS.GOOD_TIME_IMPORTANT", "WVS.SUCCESS_IMPORTANT", "WVS.GROWTH_OVER_ENVIRONMENT", "WVS.COMPETITION_NOT_HARMFUL", "WVS.HARD_WORK_LEADS_TO_SUCCESS"], "exact", "gap", "WVS programmatic route pending"),
    ("GEM", "GEM fear of failure & entrepreneurship", "GEM", ["GEM.FEAR_OF_FAILURE", "GEM.ENTREPRENEURSHIP_PREVALENCE"], "exact", "gap", "GEM national programmatic feed pending"),
    ("GFCI", "Financial Center Index", "Z/Yen", ["GFCI.SCORE"], "exact", "gap", "GFCI city scores feed pending"),
    ("VACATION", "Vacation days", "ILO / National", ["VACATION.DAYS"], "exact", "gap", "Cross-country vacation feed pending"),
    ("RETIRE", "Effective retirement age", "OECD", ["OECD.EFFECTIVE_RETIREMENT_AGE"], "exact", "gap", "OECD SDMX pension age feed pending"),
    ("EPL", "Employment protection rigidity", "OECD", ["OECD.EPL"], "proxy", "gap", "OECD EPL feed pending"),
    ("WAGE", "Wages by education cohort", "ILO / OECD", ["WAGE.EDUCATED_HOURLY_USD"], "exact", "gap", "Interim GDP_PC_COST used per §15.1"),
    ("RATING", "Sovereign credit rating", "Rating agencies", ["RATING.SOVEREIGN"], "exact", "gap", "Rating agency data paywalled"),
    ("DEBT_FOR", "Foreign-held debt share", "IMF / National", ["DEBT.FOREIGN_HELD_SHARE"], "exact", "gap", "Foreign held debt series unavailable"),
    ("VC", "Venture capital funding", "VentureSource", ["VC.TOTAL_FUNDING"], "exact", "gap", "VC total funding paywalled"),
    ("INFRA", "Infrastructure quality index", "WEF", ["INFRA.QUALITY_INDEX"], "exact", "gap", "WEF infra quality paywalled"),
    ("BIS_FX", "BIS FX turnover share & debt securities", "BIS", ["BIS.FX_TURNOVER_SHARE", "BIS.INTL_DEBT_CCY_SHARE"], "exact", "gap", "BIS bulk series pending"),
    ("WIPO_GII", "WIPO GII overall score", "WIPO", ["WIPO.GII_SCORE"], "exact", "gap", "WIPO GII indicators pending"),
    ("OECD_SAV", "OECD household savings rate", "OECD", ["OECD.HH_SAVINGS_RATE"], "exact", "live", None),
    ("ILO_CB", "Collective bargaining coverage", "ILO", ["ILO.COLLECTIVE_BARGAINING_COVERAGE"], "exact", "live", None),
    ("ILO_MINW", "Minimum wage ratio", "ILO", ["ILO.MIN_WAGE_RATIO"], "exact", "gap", "ILO min wage ratio feed pending"),
    ("COFER", "IMF COFER currency share", "IMF", ["IMF.COFER_SHARE"], "exact", "live", None),
    ("WHR_AFF", "WHR social support & positive affect", "WHR", ["WHR.SOCIAL_SUPPORT", "WHR.POSITIVE_AFFECT"], "substitute", "live", None),
    ("COVID", "COVID excess mortality", "OWID", ["COVID.EXCESS_MORTALITY_PSCORE"], "substitute", "live", None),
]

ledger_rows = []
for item, concept, pdf_src, codes, tier, status, reason in LEDGER_DATA:
    target_c_max = 0
    w_cov_max = 0.0
    n_c_max = 0
    
    if codes and "indicator_code" in df_master.columns and not df_master.empty:
        for c in codes:
            subset = df_master[df_master["indicator_code"] == c]
            if not subset.empty:
                reporters = set(subset["country_code"].unique())
                n_c_max = max(n_c_max, len(reporters))
                target_c_max = max(target_c_max, len(reporters & TARGET_SET))
                w_cov_max = max(w_cov_max, OBS_WEIGHTS[list(reporters)].sum() * 100.0)

    ledger_rows.append({
        "item": item,
        "concept": concept,
        "pdf_source": pdf_src,
        "our_codes": ", ".join(codes),
        "tier": tier,
        "global_countries": n_c_max,
        "target_countries": target_c_max,
        "weighted_cov_%": round(w_cov_max, 1),
        "status": status,
        "reason": reason or ""
    })
df_ledger = pd.DataFrame(ledger_rows)
df_ledger.to_csv(OUT_DIR / "pdf_ledger.csv", index=False)

# 3. Series Registry (FRD §7.1, Gate 2)
reg_rows = []
if "indicator_code" in df_master.columns and not df_master.empty:
    for code, g in df_master.groupby("indicator_code"):
        src = g["source"].iloc[0] if "source" in g.columns else "UNKNOWN"
        all_c = set(g["country_code"].unique())
        f_y = int(g["year"].min())
        l_y = int(g["year"].max())
        in_t = code in code_leaf_map
        in_d = code.startswith("DERIVED.")
        
        if in_t:
            status = "wired"
        elif in_d:
            status = "stand_in"
        elif code.startswith("NDGAIN."):
            status = "pdf_listed_unwired"
        else:
            status = "no_consumer"
            
        reg_rows.append({
            "code": code,
            "source": src,
            "global_countries": len(all_c),
            "target_countries": len(all_c & TARGET_SET),
            "weighted_cov_%": round(OBS_WEIGHTS[list(all_c)].sum() * 100.0, 1),
            "first_year": f_y,
            "last_year": l_y,
            "in_tree": in_t,
            "in_derived": in_d,
            "status": status
        })
df_registry = pd.DataFrame(reg_rows) if reg_rows else pd.DataFrame()
df_registry.to_csv(OUT_DIR / "df_registry.csv", index=False)

# 4. Alt Coverage (FRD §7.1 - Enriched with Target Priority)
cov_rows = []
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for comp in comps:
            for a_idx, a in enumerate(comp.get("alts", [])):
                g = df_master[df_master["indicator_code"] == a.code] if ("indicator_code" in df_master.columns and not df_master.empty) else pd.DataFrame()
                reporters = set(g["country_code"].unique()) if not g.empty else set()
                l_y = int(g[~g["is_projection"]]["year"].max()) if (not g.empty and (~g["is_projection"]).any()) else np.nan
                
                is_gap = (a.status == "GAP")
                dead = (len(reporters) == 0 and not is_gap)
                stale = (pd.notna(l_y) and l_y < STALE_BEFORE and not is_gap)
                proxy = (a.tier == "proxy_concept")
                
                cov_rows.append({
                    "leaf": leaf_path,
                    "component": comp["name"],
                    "alt_index": a_idx,
                    "code": a.code,
                    "tier": a.tier,
                    "global_countries": len(reporters),
                    "target_countries": len(reporters & TARGET_SET),
                    "weighted_cov_%": round(OBS_WEIGHTS[list(reporters)].sum() * 100.0, 1),
                    "latest_actual_year": l_y,
                    "row_count": len(g),
                    "dead": dead,
                    "stale": stale,
                    "proxy": proxy,
                    "gap": is_gap
                })
df_alt_cov = pd.DataFrame(cov_rows)
df_alt_cov.to_csv(OUT_DIR / "alt_coverage.csv", index=False)

# 5. Effective Weights (FRD §7.1)
eff_rows = []
for leaf_path, comps, leaf_w in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        gauge = leaf_path.split("/")[0]
        for comp in comps:
            c_w = comp.get("weight", 1.0)
            alts = comp.get("alts", [])
            if alts:
                first_alt = alts[0]
                eff_rows.append({
                    "gauge": gauge,
                    "leaf_path": leaf_path,
                    "component_name": comp["name"],
                    "code": first_alt.code,
                    "effective_weight": leaf_w * c_w,
                    "tier": first_alt.tier,
                    "status": first_alt.status or "live"
                })
df_eff = pd.DataFrame(eff_rows)
df_eff.to_csv(OUT_DIR / "effective_weights.csv", index=False)

# 6. Direction Map (FRD §6.5)
dir_rows = []
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for c in comps:
            for a in c.get("alts", []):
                ambiguous = False
                if any(k in a.code for k in ["CPI", "INFLATION", "CBPOL", "TRFT", "TOTL.GD.ZS", "EPL", "COLLECTIVE"]):
                    ambiguous = True
                if "cost" in leaf_path.lower() and "GDP" in a.code:
                    ambiguous = True
                dir_rows.append({
                    "code": a.code,
                    "leaf_path": leaf_path,
                    "direction": a.direction,
                    "ambiguous": ambiguous,
                    "tier": a.tier
                })
df_dir = pd.DataFrame(dir_rows).drop_duplicates(subset=["code", "leaf_path"])
df_dir.to_csv(OUT_DIR / "direction_map.csv", index=False)

# 7. Switch Report (FRD §7.1 - Enriched with Core vs Peripheral Split)
if not df_comp_scores.empty and "observed" in df_comp_scores.columns and df_comp_scores["observed"].any():
    obs = df_comp_scores[df_comp_scores["observed"]].copy()
    switch_counts = obs.groupby(["leaf_path", "component_name", "alt_used"]).size().rename("country_years").reset_index()
    
    # Identify switched countries and segment by target portfolio
    c_alts = obs.groupby(["leaf_path", "component_name", "country_code"])["alt_used"].nunique()
    switched_mask = c_alts > 1
    switched_df = switched_mask.reset_index().rename(columns={"alt_used": "has_switched"})
    switched_df = switched_df[switched_df["has_switched"]]
    switched_df["is_target"] = switched_df["country_code"].isin(TARGET_SET)
    
    switch_summary = switched_df.groupby(["leaf_path", "component_name"]).agg(
        total_switched=("country_code", "count"),
        target_switched=("is_target", "sum")
    ).reset_index()
    
    df_switch = pd.merge(switch_counts, switch_summary, on=["leaf_path", "component_name"], how="left")
    df_switch["total_switched"] = df_switch["total_switched"].fillna(0).astype(int)
    df_switch["target_switched"] = df_switch["target_switched"].fillna(0).astype(int)
else:
    df_switch = pd.DataFrame(columns=["leaf_path", "component_name", "alt_used", "country_years", "total_switched", "target_switched"])
df_switch.to_csv(OUT_DIR / "switch_report.csv", index=False)

# 8. Agreement Report (FRD §7.1 - Target & Global Correlation)
agreement_rows = []
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for comp in comps:
            alts = comp.get("alts", [])
            if len(alts) > 1:
                for i in range(len(alts)):
                    for j in range(i + 1, len(alts)):
                        c1, c2 = alts[i].code, alts[j].code
                        if c1 in wide_map and c2 in wide_map:
                            s1, s2 = wide_map[c1], wide_map[c2]
                            aligned = pd.concat([s1, s2], axis=1, keys=["v1", "v2"]).dropna()
                            n_overlap = len(aligned)
                            
                            # Filter to targets to eliminate micro-state noise
                            aligned_target = aligned[aligned.index.get_level_values("country_code").isin(TARGET_SET)]
                            n_target_overlap = len(aligned_target)
                            
                            corr_global = spearmanr(aligned["v1"], aligned["v2"])[0] if n_overlap >= 5 else np.nan
                            corr_target = spearmanr(aligned_target["v1"], aligned_target["v2"])[0] if n_target_overlap >= 5 else np.nan
                            ratio = float(np.median(aligned["v1"] / aligned["v2"].replace(0, np.nan).dropna())) if n_overlap >= 5 else np.nan
                            
                            agreement_rows.append({
                                "leaf_path": leaf_path,
                                "component_name": comp["name"],
                                "alt_1": c1,
                                "alt_2": c2,
                                "overlap_count": n_overlap,
                                "target_overlap_count": n_target_overlap,
                                "spearman_global": round(corr_global, 3) if pd.notna(corr_global) else np.nan,
                                "spearman_target": round(corr_target, 3) if pd.notna(corr_target) else np.nan,
                                "median_ratio": ratio
                            })
df_agreement = pd.DataFrame(agreement_rows) if agreement_rows else pd.DataFrame()
df_agreement.to_csv(OUT_DIR / "agreement_report.csv", index=False)

# 9. Proxy Candidates Report (FRD §7.1 - Weighted Value Added)
proxy_rows = []
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for comp in comps:
            alts = comp.get("alts", [])
            if len(alts) > 1:
                for a in alts[1:]:
                    if a.tier == "proxy_concept" and a.code in wide_map:
                        sub_comp = df_comp_scores[(df_comp_scores["leaf_path"] == leaf_path) & (df_comp_scores["component_name"] == comp["name"])]
                        missing = sub_comp[~sub_comp["observed"]][["year", "country_code"]]
                        proxy_series = wide_map[a.code]
                        
                        added_pairs = [
                            (row["year"], row["country_code"])
                            for _, row in missing.iterrows()
                            if (row["year"], row["country_code"]) in proxy_series.index
                        ]
                        
                        total_added = len(added_pairs)
                        target_added = sum(1 for _, c in added_pairs if c in TARGET_SET)
                        weighted_impact = sum(OBS_WEIGHTS.get(c, 0.0) for _, c in added_pairs)
                        
                        proxy_rows.append({
                            "leaf_path": leaf_path,
                            "component_name": comp["name"],
                            "proxy_code": a.code,
                            "country_years_added": total_added,
                            "target_years_added": target_added,
                            "weighted_years_added": round(weighted_impact, 2)
                        })
df_proxy = pd.DataFrame(proxy_rows) if proxy_rows else pd.DataFrame()
df_proxy.to_csv(OUT_DIR / "proxy_candidates.csv", index=False)

print("All QA reports and PDF ledger successfully generated!")
print(f"  df_health: {len(df_health)} rows | pdf_ledger: {len(df_ledger)} rows | df_registry: {len(df_registry)} rows")
print(f"  alt_coverage: {len(df_alt_cov)} rows | effective_weights: {len(df_eff)} rows | direction_map: {len(df_dir)} rows")

In [ ]:
# 7a. QA Reports & PDF Ledger
# 1. df_health (FRD §7.1)
health_rows = []
ppp_df = df_master[df_master["indicator_code"].isin(["IMF.PPPGDP", "NY.GDP.MKTP.PP.KD"])].copy() if "indicator_code" in df_master.columns else pd.DataFrame()
if not ppp_df.empty:
    latest_ppp = ppp_df.sort_values("year").groupby("country_code")["value"].last()
    total_ppp = latest_ppp.sum()
    ppp_weights = (latest_ppp / total_ppp) if total_ppp > 0 else pd.Series(1.0 / len(UNIVERSE), index=UNIVERSE)
else:
    ppp_weights = pd.Series(1.0 / len(UNIVERSE), index=UNIVERSE)

code_leaf_map = defaultdict(list)
code_tier_map = {}
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for c in comps:
            for a in c.get("alts", []):
                code_leaf_map[a.code].append(leaf_path)
                code_tier_map[a.code] = a.tier

if "indicator_code" in df_master.columns and not df_master.empty:
    for code, g in df_master.groupby("indicator_code"):
        n_c = g["country_code"].nunique()
        cov_pct = (n_c / len(UNIVERSE)) * 100.0
        present_countries = set(g["country_code"].unique())
        w_cov_pct = sum(ppp_weights.get(c, 0.0) for c in present_countries) * 100.0
        latest_y = int(g["year"].max())
        stale = latest_y < STALE_BEFORE
        
        counts_by_year = g.groupby("year")["country_code"].nunique()
        valid_years = counts_by_year[counts_by_year >= MIN_CROSS_SECTION].index
        first_valid_y = int(valid_years.min()) if len(valid_years) > 0 else np.nan
        
        std_val = g["value"].std()
        degenerate = (std_val == 0.0 or pd.isna(std_val))
        is_proj = bool(g["is_projection"].any())
        leaves = code_leaf_map.get(code, [])
        gauge = leaves[0].split("/")[0] if leaves else "unassigned"
        tier = code_tier_map.get(code, "exact")
        
        health_rows.append({
            "gauge": gauge,
            "indicator": code,
            "source": g["source"].iloc[0] if "source" in g.columns else "UNKNOWN",
            "reporting_countries": n_c,
            "coverage_%": round(cov_pct, 1),
            "weighted_cov_%": round(w_cov_pct, 1),
            "latest_year": latest_y,
            "stale": stale,
            "first_valid_year": first_valid_y,
            "degenerate": degenerate,
            "is_projection": is_proj,
            "tier": tier,
            "status": "wired" if leaves else "pdf_listed_unwired"
        })

df_health = pd.DataFrame(health_rows) if health_rows else pd.DataFrame(columns=["gauge", "indicator", "source", "reporting_countries", "coverage_%", "weighted_cov_%", "latest_year", "stale", "first_valid_year", "degenerate", "is_projection", "tier", "status"])
df_health.to_csv(OUT_DIR / "df_health.csv", index=False)

# 2. PDF Ledger (FRD §13, §11, Gate 1)
LEDGER_DATA = [
(1, "Access to essential health services", "Social Progress Imperative", ["SH.UHC.SRVS.CV.XD"], "substitute", "substitute", None),
    (2, "Aerospace and defense sector returns", "Thomson Reuters", [], "exact", "gap", "Paywalled"),
    (3, "Agriculture, forestry, fishing value added", "WDI", ["NV.AGR.TOTL.ZS", "NV.AGR.TOTL.CD"], "exact", "live", None),
    (4, "AI: new companies, patents, citations", "CSET", ["CSET.AI_OUTPUT"], "exact", "skipped", "CSET skipped per D4"),
    (5, "Banking claims by currency and source country", "BIS", ["BIS.LBS.CROSSBORDER_CLAIMS"], "exact", "gap", "BIS claims bulk file pending"),
    (6, "Bloomberg Innovation Index", "Bloomberg", ["BLOOMBERG.INNOVATION_INDEX"], "exact", "gap", "Paywalled"),
    (7, "Book titles per capita", "CLIO Infra", [], "exact", "gap", "Historical dataset, no current programmatic consumer"),
    (8, "Child mortality rate", "Social Progress Imperative", ["SH.DYN.MORT"], "substitute", "substitute", None),
    (9, "Climate vulnerability and preparedness, subcomponents", "ND-GAIN", ["NDGAIN.VULNERABILITY", "NDGAIN.READINESS"], "proxy", "live", None),
    (10, "CO2 emissions", "CLIO Infra", ["EN.GHG.CO2.MT.CE.AR5", "EN.GHG.CO2.PC.CE.AR5"], "substitute", "live", None),
    (11, "Conflict intensity and count per capita", "Cline Center", ["CLINE.CONFLICT_PER_CAPITA"], "exact", "gap", "Cline Center public API pending"),
    (12, "Construction permits score", "WB Doing Business", ["IC.CNST.PRMT.DFRN.DB1619", "IC.CNST.PRMT.DFRN.DB0615"], "exact", "live", None),
    (13, "Days lost to labor disputes", "CLIO Infra", ["ILO.STRIKES_DAYS_NOT_WORKED"], "substitute", "substitute", None),
    (14, "Discontent, populism, conflict terms in media", "GDELT", ["GDELT.DISCONTENT_TERMS"], "exact", "gap", "GDELT API pending"),
    (15, "Education spending", "WDI", ["SE.XPD.TOTL.GD.ZS"], "exact", "live", None),
    (16, "Educational attainment", "Barro-Lee", ["BAR.SCHL.25UP"], "exact", "live", None),
    (17, "Electricity production", "NBER CHAT", ["OWID.ELECTRICITY_GEN_TWH", "EIA.ELECTRICITY.GENERATION.BKWH"], "substitute", "substitute", None),
    (18, "Energy production and consumption", "EIA", ["EIA.ENERGY.PRODUCTION_QBTU", "EIA.ENERGY.SELF_SUFF"], "exact", "live", None),
    (19, "Enforcing contracts score", "WB Doing Business", ["ENF.CONT.COEN.DB1719.DFRN", "ENF.CONT.COEN.DB0415"], "exact", "live", None),
    (20, "GINI coefficient", "WDI", ["SI.POV.GINI", "PIP.GINI"], "exact", "live", None),
    (21, "Google searches for political conflict terms", "Google Trends", [], "exact", "skipped", "Google Trends skipped per D4"),
    (22, "Google searches indicating discontent", "Google Trends", [], "exact", "skipped", "Google Trends skipped per D4"),
    (23, "Government disapproval, verbal opposition in news", "GDELT", ["GDELT.GOVT_DISAPPROVAL"], "exact", "gap", "GDELT API pending"),
    (24, "Gross expenditure on R&D", "WIPO GII", ["GB.XPD.RSDV.GD.ZS"], "substitute", "substitute", None),
    (25, "Historical aviation passengers and cargo", "NBER CHAT", ["IS.AIR.PSGR", "IS.AIR.GOOD.MT.K1"], "substitute", "substitute", None),
    (26, "Historical technology adoption", "NBER CHAT", [], "exact", "gap", "Historical dataset, no 2024 gauge consumer"),
    (27, "Homicide rate", "Social Progress Imperative", ["VC.IHR.PSRC.P5"], "substitute", "substitute", None),
    (28, "Hours worked per week", "World Inequality Database", ["ILO.HOURS_WEEKLY_MODELLED_MALE", "ILO.HOURS_WEEKLY_MODELLED", "DERIVED.PWT_HOURS_WEEKLY"], "exact", "live", None),
    (29, "Human Capital Index", "World Bank", ["HD.HCI.OVRL", "HD.HCI.EYRS", "HD.HCI.HLOS"], "exact", "live", None),
    (30, "Income distributions", "World Bank", ["SI.DST.FRST.20", "SI.DST.05TH.20", "SI.DST.10TH.10", "PIP.DECILE1", "PIP.DECILE10"], "exact", "live", None),
    (31, "Internal armed conflict", "CLIO Infra", ["UCDP.INTERNAL_CONFLICT_DEATHS"], "substitute", "substitute", None),
    (32, "Internet users", "NBER CHAT", ["IT.NET.USER.ZS"], "substitute", "substitute", None),
    (33, "Male labor force participation", "World Bank", ["SL.TLF.CACT.MA.ZS"], "exact", "live", None),
    (34, "Metals and ores production", "CLIO Infra", ["USGS.METALS_PRODUCTION"], "exact", "gap", "USGS metals pending"),
    (35, "Nuclear weapon stockpile", "Federation of American Scientists", ["FAS.NUCLEAR_STOCKPILE"], "exact", "live", None),
    (36, "Number of new patents", "WIPO GII", ["IP.PAT.RESD"], "substitute", "substitute", None),
    (37, "Numeracy", "CLIO Infra", [], "exact", "gap", "Historical dataset, no current consumer"),
    (38, "OTC FX turnover", "BIS", ["BIS.FX_TURNOVER_VALUE"], "exact", "gap", "BIS triennial FX survey programmatic feed pending"),
    (39, "Particulate matter pollution", "Social Progress Imperative", ["EN.ATM.PM25.MC.M3"], "substitute", "substitute", None),
    (40, "Patent applications", "WDI", ["IP.PAT.RESD", "IP.PAT.NRES"], "exact", "live", None),
    (41, "People displaced by conflict and violence", "WDI", ["UNHCR.DISPLACED_PCT_POP"], "substitute", "substitute", None),
    (42, "People using safely managed drinking water", "World Bank", ["SH.H2O.SMDW.ZS"], "exact", "live", None),
    (43, "Prevalence of undernourishment", "World Bank", ["SN.ITK.DEFC.ZS"], "exact", "live", None),
    (44, "Protecting investors value score", "WB Doing Business", ["PROT.MINOR.INV.DFRN.DB1519", "PROT.MINOR.INV.DFRN.DB0614"], "exact", "live", None),
    (45, "Protest intensity index", "GDELT", ["GDELT.PROTEST_INTENSITY"], "exact", "gap", "GDELT API pending"),
    (46, "Public sector corruption and theft", "V-Dem", ["VDEM.v2exthftps", "VDEM.v2excrptps"], "exact", "live", None),
    (47, "Rail lines opened and rail cargo", "NBER CHAT", ["IS.RRS.TOTL.KM", "IS.RRS.GOOD.MT.K6"], "substitute", "substitute", None),
    (48, "Real GDP per worker", "Penn World Table", ["PWT.RGDPO_PER_WORKER"], "exact", "live", None),
    (49, "Real growth rate", "World Bank", ["NY.GDP.MKTP.KD.ZG", "IMF.NGDP_RPCH"], "exact", "live", None),
    (50, "Renewable freshwater resources", "WDI", ["ER.H2O.INTR.K3", "ER.H2O.INTR.PC"], "exact", "live", None),
    (51, "Rigorous and impartial government score", "V-Dem", ["VDEM.v2clrspct"], "exact", "live", None),
    (52, "Royalty and license fees", "WIPO GII", ["BM.GSR.ROYL.CD", "BX.GSR.ROYL.CD"], "substitute", "substitute", None),
    (53, "Share of universities in top 500", "Shanghai Ranking", ["ARWU.TOP100_COUNT", "ARWU.TOP500_COUNT"], "exact", "live", None),
    (54, "Shipping tonnage by sail, steam, motor and total", "NBER CHAT", ["IS.SHP.GCNW.XQ"], "proxy", "proxy", None),
    (55, "Starting a business score", "WB Doing Business", ["IC.REG.DFRN.PC.DFRN"], "exact", "live", None),
    (56, "Suicide mortality rate", "World Bank", ["SH.STA.SUIC.P5"], "exact", "live", None),
    (57, "Tariff rate", "World Bank", ["TM.TAX.MRCH.WM.AR.ZS"], "exact", "live", None),
    (58, "Top 1% and top 10% income shares", "World Inequality Database", ["WID.TOP1_INCOME", "WID.TOP10_INCOME"], "exact", "live", None),
    (59, "Top 1% and top 10% wealth shares", "World Inequality Database", ["WID.TOP1_WEALTH", "WID.TOP10_WEALTH"], "exact", "live", None),
    (60, "Total labor disputes", "CLIO Infra", ["ILO.STRIKES_COUNT"], "substitute", "substitute", None),
    (61, "Transparent laws with predictable enforcement", "V-Dem", ["VDEM.v2cltrnslw"], "exact", "live", None),
    (62, "Universities founded", "CLIO Infra", [], "exact", "gap", "Historical dataset, no current consumer"),
    (63, "Number of new businesses, people creating businesses", "WIPO GII", ["IC.BUS.NREG", "IC.BUS.NDNS.ZS"], "substitute", "substitute", None),
    (64, "Number of new major websites", "WIPO GII", ["WIPO.NEW_MAJOR_WEBSITES"], "exact", "gap", "WIPO web metrics feed pending"),
    # Non-list inputs
    ("WVS", "World Values Survey items (8)", "World Values Survey", ["WVS.HAPPY", "WVS.CHILDREN_RESPONSIBILITY", "WVS.GROWTH_IMPORTANT", "WVS.GOOD_TIME_IMPORTANT", "WVS.SUCCESS_IMPORTANT", "WVS.GROWTH_OVER_ENVIRONMENT", "WVS.COMPETITION_NOT_HARMFUL", "WVS.HARD_WORK_LEADS_TO_SUCCESS"], "exact", "gap", "WVS programmatic route pending"),
    ("GEM", "GEM fear of failure & entrepreneurship", "GEM", ["GEM.FEAR_OF_FAILURE", "GEM.ENTREPRENEURSHIP_PREVALENCE"], "exact", "gap", "GEM national programmatic feed pending"),
    ("GFCI", "Financial Center Index", "Z/Yen", ["GFCI.SCORE"], "exact", "gap", "GFCI city scores feed pending"),
    ("VACATION", "Vacation days", "ILO / National", ["VACATION.DAYS"], "exact", "gap", "Cross-country vacation feed pending"),
    ("RETIRE", "Effective retirement age", "OECD", ["OECD.EFFECTIVE_RETIREMENT_AGE"], "exact", "gap", "OECD SDMX pension age feed pending"),
    ("EPL", "Employment protection rigidity", "OECD", ["OECD.EPL"], "proxy", "gap", "OECD EPL feed pending"),
    ("WAGE", "Wages by education cohort", "ILO / OECD", ["WAGE.EDUCATED_HOURLY_USD"], "exact", "gap", "Interim GDP_PC_COST used per §15.1"),
    ("RATING", "Sovereign credit rating", "Rating agencies", ["RATING.SOVEREIGN"], "exact", "gap", "Rating agency data paywalled"),
    ("DEBT_FOR", "Foreign-held debt share", "IMF / National", ["DEBT.FOREIGN_HELD_SHARE"], "exact", "gap", "Foreign held debt series unavailable"),
    ("VC", "Venture capital funding", "VentureSource", ["VC.TOTAL_FUNDING"], "exact", "gap", "VC total funding paywalled"),
    ("INFRA", "Infrastructure quality index", "WEF", ["INFRA.QUALITY_INDEX"], "exact", "gap", "WEF infra quality paywalled"),
    ("BIS_FX", "BIS FX turnover share & debt securities", "BIS", ["BIS.FX_TURNOVER_SHARE", "BIS.INTL_DEBT_CCY_SHARE"], "exact", "gap", "BIS bulk series pending"),
    ("WIPO_GII", "WIPO GII overall score", "WIPO", ["WIPO.GII_SCORE"], "exact", "gap", "WIPO GII indicators pending"),
    ("OECD_SAV", "OECD household savings rate", "OECD", ["OECD.HH_SAVINGS_RATE"], "exact", "live", None),
    ("ILO_CB", "Collective bargaining coverage", "ILO", ["ILO.COLLECTIVE_BARGAINING_COVERAGE"], "exact", "live", None),
    ("ILO_MINW", "Minimum wage ratio", "ILO", ["ILO.MIN_WAGE_RATIO"], "exact", "gap", "ILO min wage ratio feed pending"),
    ("COFER", "IMF COFER currency share", "IMF", ["IMF.COFER_SHARE"], "exact", "live", None),
    ("WHR_AFF", "WHR social support & positive affect", "WHR", ["WHR.SOCIAL_SUPPORT", "WHR.POSITIVE_AFFECT"], "substitute", "live", None),
    ("COVID", "COVID excess mortality", "OWID", ["COVID.EXCESS_MORTALITY_PSCORE"], "substitute", "live", None),
]

ledger_rows = []
for item, concept, pdf_src, codes, tier, status, reason in LEDGER_DATA:
    n_c = max([df_master[df_master["indicator_code"] == c]["country_code"].nunique() for c in codes], default=0) if (codes and "indicator_code" in df_master.columns and not df_master.empty) else 0
    ledger_rows.append({
        "item": item,
        "concept": concept,
        "pdf_source": pdf_src,
        "our_codes": ", ".join(codes),
        "tier": tier,
        "n_countries": n_c,
        "status": status,
        "reason": reason or ""
    })
df_ledger = pd.DataFrame(ledger_rows)
df_ledger.to_csv(OUT_DIR / "pdf_ledger.csv", index=False)

# 3. Series Registry (FRD §7.1, Gate 2)
reg_rows = []
if "indicator_code" in df_master.columns and not df_master.empty:
    for code, g in df_master.groupby("indicator_code"):
        src = g["source"].iloc[0] if "source" in g.columns else "UNKNOWN"
        n_c = g["country_code"].nunique()
        f_y = int(g["year"].min())
        l_y = int(g["year"].max())
        in_t = code in code_leaf_map
        in_d = code.startswith("DERIVED.")
        if in_t:
            status = "wired"
        elif in_d:
            status = "stand_in"
        elif code.startswith("NDGAIN."):
            status = "pdf_listed_unwired"
        else:
            status = "no_consumer"
        reg_rows.append({
            "code": code,
            "source": src,
            "n_countries": n_c,
            "first_year": f_y,
            "last_year": l_y,
            "in_tree": in_t,
            "in_derived": in_d,
            "status": status
        })
df_registry = pd.DataFrame(reg_rows) if reg_rows else pd.DataFrame(columns=["code", "source", "n_countries", "first_year", "last_year", "in_tree", "in_derived", "status"])
df_registry.to_csv(OUT_DIR / "df_registry.csv", index=False)

# 4. Alt Coverage (FRD §7.1)
cov_rows = []
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for comp in comps:
            for a_idx, a in enumerate(comp.get("alts", [])):
                g = df_master[df_master["indicator_code"] == a.code] if ("indicator_code" in df_master.columns and not df_master.empty) else pd.DataFrame()
                n_c = g["country_code"].nunique() if not g.empty else 0
                l_y = int(g["year"].max()) if not g.empty else np.nan
                cnt = len(g)
                is_gap = (a.status == "GAP")
                dead = (n_c == 0 and not is_gap)
                stale = (pd.notna(l_y) and l_y < STALE_BEFORE and not is_gap)
                proxy = (a.tier == "proxy_concept")
                cov_rows.append({
                    "leaf": leaf_path,
                    "component": comp["name"],
                    "alt_index": a_idx,
                    "code": a.code,
                    "tier": a.tier,
                    "countries_covered": n_c,
                    "latest_year": l_y,
                    "row_count": cnt,
                    "dead": dead,
                    "stale": stale,
                    "proxy": proxy,
                    "gap": is_gap
                })
df_alt_cov = pd.DataFrame(cov_rows)
df_alt_cov.to_csv(OUT_DIR / "alt_coverage.csv", index=False)

# 5. Effective Weights (FRD §7.1)
eff_rows = []
for leaf_path, comps, leaf_w in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        gauge = leaf_path.split("/")[0]
        for comp in comps:
            c_w = comp.get("weight", 1.0)
            alts = comp.get("alts", [])
            if alts:
                first_alt = alts[0]
                eff_rows.append({
                    "gauge": gauge,
                    "leaf_path": leaf_path,
                    "component_name": comp["name"],
                    "code": first_alt.code,
                    "effective_weight": leaf_w * c_w,
                    "tier": first_alt.tier,
                    "status": first_alt.status or "live"
                })
df_eff = pd.DataFrame(eff_rows)
df_eff.to_csv(OUT_DIR / "effective_weights.csv", index=False)

# 6. Direction Map (FRD §6.5)
dir_rows = []
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for c in comps:
            for a in c.get("alts", []):
                ambiguous = False
                if any(k in a.code for k in ["CPI", "INFLATION", "CBPOL", "TRFT", "TOTL.GD.ZS", "EPL", "COLLECTIVE"]):
                    ambiguous = True
                if "cost" in leaf_path.lower() and "GDP" in a.code:
                    ambiguous = True
                dir_rows.append({
                    "code": a.code,
                    "leaf_path": leaf_path,
                    "direction": a.direction,
                    "ambiguous": ambiguous,
                    "tier": a.tier
                })
df_dir = pd.DataFrame(dir_rows).drop_duplicates(subset=["code", "leaf_path"])
df_dir.to_csv(OUT_DIR / "direction_map.csv", index=False)

# 7. Switch Report (FRD §7.1)
if not df_comp_scores.empty and "observed" in df_comp_scores.columns and df_comp_scores["observed"].any():
    obs = df_comp_scores[df_comp_scores["observed"]].copy()
    switch_counts = obs.groupby(["leaf_path", "component_name", "alt_used"]).size().rename("country_years").reset_index()
    c_alts = obs.groupby(["leaf_path", "component_name", "country_code"])["alt_used"].nunique()
    switched_countries = (c_alts > 1).groupby(["leaf_path", "component_name"]).sum().rename("countries_switched").reset_index()
    df_switch = pd.merge(switch_counts, switched_countries, on=["leaf_path", "component_name"], how="left")
else:
    df_switch = pd.DataFrame(columns=["leaf_path", "component_name", "alt_used", "country_years", "countries_switched"])
df_switch.to_csv(OUT_DIR / "switch_report.csv", index=False)

# 8. Agreement Report (FRD §7.1)
wide_map = {code: g.set_index(["year", "country_code"])["value"] for code, g in df_master.groupby("indicator_code")} if ("indicator_code" in df_master.columns and not df_master.empty) else {}
agreement_rows = []
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for comp in comps:
            alts = comp.get("alts", [])
            if len(alts) > 1:
                for i in range(len(alts)):
                    for j in range(i + 1, len(alts)):
                        c1, c2 = alts[i].code, alts[j].code
                        if c1 in wide_map and c2 in wide_map:
                            s1, s2 = wide_map[c1], wide_map[c2]
                            aligned = pd.concat([s1, s2], axis=1, keys=["v1", "v2"]).dropna()
                            n_overlap = len(aligned)
                            if n_overlap >= 5:
                                corr, _ = spearmanr(aligned["v1"], aligned["v2"])
                                ratio = float(np.median(aligned["v1"] / aligned["v2"].replace(0, np.nan).dropna()))
                            else:
                                corr, ratio = np.nan, np.nan
                            agreement_rows.append({
                                "leaf_path": leaf_path,
                                "component_name": comp["name"],
                                "alt_1": c1,
                                "alt_2": c2,
                                "overlap_count": n_overlap,
                                "spearman_corr": corr,
                                "median_ratio": ratio
                            })
df_agreement = pd.DataFrame(agreement_rows) if agreement_rows else pd.DataFrame(columns=["leaf_path", "component_name", "alt_1", "alt_2", "overlap_count", "spearman_corr", "median_ratio"])
df_agreement.to_csv(OUT_DIR / "agreement_report.csv", index=False)

# 9. Proxy Candidates Report (FRD §7.1)
proxy_rows = []
for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, list):
        for comp in comps:
            alts = comp.get("alts", [])
            if len(alts) > 1:
                for a_idx, a in enumerate(alts[1:], start=1):
                    if a.tier == "proxy_concept" and a.code in wide_map:
                        sub_comp = df_comp_scores[(df_comp_scores["leaf_path"] == leaf_path) & (df_comp_scores["component_name"] == comp["name"])]
                        missing_idx = sub_comp[~sub_comp["observed"]][["year", "country_code"]]
                        proxy_series = wide_map[a.code]
                        added = sum(1 for _, row in missing_idx.iterrows() if (row["year"], row["country_code"]) in proxy_series.index)
                        proxy_rows.append({
                            "leaf_path": leaf_path,
                            "component_name": comp["name"],
                            "proxy_code": a.code,
                            "country_years_added_if_allowed": added
                        })
df_proxy = pd.DataFrame(proxy_rows) if proxy_rows else pd.DataFrame(columns=["leaf_path", "component_name", "proxy_code", "country_years_added_if_allowed"])
df_proxy.to_csv(OUT_DIR / "proxy_candidates.csv", index=False)

print("All QA reports and PDF ledger successfully generated!")
print(f"  df_health: {len(df_health)} rows | pdf_ledger: {len(df_ledger)} rows | df_registry: {len(df_registry)} rows")
print(f"  alt_coverage: {len(df_alt_cov)} rows | effective_weights: {len(df_eff)} rows | direction_map: {len(df_dir)} rows")


# 7b. Inline Asserts & Hard Gates (§7.2, §7.3, §14)
Enforces all Hard Gates (Definition of Done) and structure checks:
- **Gate 1 (Ledger Gate)**: Every item has status; every gap/skipped row has a reason.
- **Gate 2 (Registry Gate)**: `no_consumer` is empty; no dead alternative outside GAP.
- **Gate 3 (Weight Gate)**: Hours-worked effective weight is 15.45% ± 0.1%; health weights sum to 1.0; happiness weights sum to 1.0; growth split 0.65/0.35; productivity split 0.70/0.30; no code in two leaves of one gauge.
- **Gate 4 (Data Sanity Gates)**:
  - 4(a) `IMF.NGDPDPC` for DALIO_35 vs §14.3 (median rel dev ≤ 10%, max ≤ 30%).
  - 4(b) `DERIVED.WORKING_AGE_GROWTH_FWD10` at 2024 for DALIO_35 vs §14.2 (≥ 90% within 0.3 pp).
- **Printed Checks (§14.1, §14.4)**: Headline Empire Scores comparison & qualitative checks.


In [ ]:
# 7b. Inline Asserts & Hard Gates (Non-halting check version)
structural_passed = True

# --- Inline Structural Asserts (FRD §7.2) ---
# 1. Synthetic Standardisation & Coalescing Scale Test
n_syn = 40
c_syn = [f"C{i:02d}" for i in range(n_syn)]
s_a = pd.Series(np.linspace(10, 500, n_syn), index=c_syn)
s_b = pd.Series(np.linspace(0.01, 0.99, n_syn), index=c_syn)

def syn_rank_pct(s):
    n = s.count()
    return (s.rank(method="average", ascending=True) - 1.0) / (n - 1.0) * 100.0

rk_a = syn_rank_pct(s_a)
rk_b = syn_rank_pct(s_b)

if not (rk_a.min() == 0.0 and rk_a.max() == 100.0):
    print(f"[FAIL] Synthetic scale test (rk_a): min={rk_a.min()}, max={rk_a.max()} (expected 0.0 and 100.0)")
    structural_passed = False

if not (rk_b.min() == 0.0 and rk_b.max() == 100.0):
    print(f"[FAIL] Synthetic scale test (rk_b): min={rk_b.min()}, max={rk_b.max()} (expected 0.0 and 100.0)")
    structural_passed = False

rk_a_sparse = rk_a.copy()
rk_a_sparse.iloc[:5] = np.nan
coalesced = rk_a_sparse.combine_first(rk_b)

if coalesced.isna().sum() != 0:
    print(f"[FAIL] Coalesced scale test: found {coalesced.isna().sum()} NaN values")
    structural_passed = False

if not (0.0 <= coalesced.min() and coalesced.max() <= 100.0):
    print(f"[FAIL] Coalesced bounds: min={coalesced.min()}, max={coalesced.max()} out of [0.0, 100.0]")
    structural_passed = False

# 2. Synthetic Renormalisation Test
comp_w = np.array([0.25, 0.25, 0.25, 0.25])
comp_s = np.array([np.nan, 60.0, 70.0, 80.0])
pres = ~np.isnan(comp_s)

w_ratio = comp_w[pres].sum() / comp_w.sum()
if not np.isclose(w_ratio, 0.75):
    print(f"[FAIL] Renormalisation test: weight ratio is {w_ratio}, expected 0.75")
    structural_passed = False

renorm_score = (comp_s[pres] * comp_w[pres]).sum() / comp_w[pres].sum()
if not np.isclose(renorm_score, 70.0):
    print(f"[FAIL] Renormalisation test: weighted score is {renorm_score}, expected 70.0")
    structural_passed = False

# 3. No Duplicate Code in Health or Happiness
for gname in ["health_index", "happiness_index"]:
    seen = set()
    for leaf_path, comps, _ in iter_leaves({gname: RAY_DALIO_GAUGE_TREES[gname]}):
        if isinstance(comps, list):
            for c in comps:
                for a in c.get("alts", []):
                    if a.code in seen:
                        print(f"[FAIL] Duplicate code {a.code} in {gname}")
                        structural_passed = False
                    seen.add(a.code)

# 4. No Code in Both Halves of One Strength Determinant
for det_name, det_node in RAY_DALIO_GAUGE_TREES["strength_determinants"]["children"].items():
    if "children" in det_node and "total" in det_node["children"] and "per_capita" in det_node["children"]:
        tot_c = {a.code for c in det_node["children"]["total"].get("components", []) for a in c.get("alts", [])}
        pc_c = {a.code for c in det_node["children"]["per_capita"].get("components", []) for a in c.get("alts", [])}
        overlap = tot_c & pc_c
        if overlap:
            print(f"[FAIL] Code overlap in determinant {det_name}: {overlap}")
            structural_passed = False

# 5. Every @ Reference Resolves
def resolve_node_ref(ref_str, trees):
    path = ref_str.lstrip("@").split(".")
    curr = trees
    for p in path:
        if not isinstance(curr, dict):
            print(f"[FAIL] Failed resolving {ref_str} at segment '{p}' (parent node is not a dict)")
            return None
        curr = curr.get("children", {}).get(p, curr.get(p))
        if curr is None:
            print(f"[FAIL] Failed resolving {ref_str} at segment '{p}'")
            return None
    return curr

for leaf_path, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(comps, dict) and "reference" in comps:
        resolved = resolve_node_ref(comps["reference"], RAY_DALIO_GAUGE_TREES)
        if resolved is None:
            print(f"[FAIL] Unresolved reference '{comps['reference']}' at leaf: {leaf_path}")
            structural_passed = False

if structural_passed:
    print("All inline structural asserts passed!")
else:
    print("[WARNING] One or more inline structural asserts failed!")

# --- Hard Gates (FRD §7.3) ---
gates_passed = True

# Gate 1: Ledger Gate
gate1_passed = True
valid_statuses = {"live", "substitute", "proxy", "gap", "skipped"}
if not df_ledger["status"].isin(valid_statuses).all():
    invalid = df_ledger.loc[~df_ledger["status"].isin(valid_statuses), "status"].unique()
    print(f"[FAIL] Gate 1: Found unexpected status values in ledger: {invalid}")
    gate1_passed = False

gaps_skipped = df_ledger[df_ledger["status"].isin({"gap", "skipped"})]
if not (gaps_skipped["reason"].str.strip().str.len() > 0).all():
    missing_reasons_count = (gaps_skipped["reason"].str.strip().str.len() == 0).sum()
    print(f"[FAIL] Gate 1: Found {missing_reasons_count} gap/skipped items without reasons!")
    gate1_passed = False

if gate1_passed:
    print("GATE 1 (Ledger Gate): PASSED!")
else:
    print("GATE 1 (Ledger Gate): FAILED!")
    gates_passed = False

# Gate 2: Registry Gate
gate2_passed = True
no_consumers = df_registry[df_registry["status"] == "no_consumer"]
if len(no_consumers) > 0:
    print(f"[FAIL] Gate 2: Found {len(no_consumers)} unconsumed series in registry!")
    gate2_passed = False

if gate2_passed:
    print("GATE 2 (Registry Gate): PASSED!")
else:
    print("GATE 2 (Registry Gate): FAILED!")
    gates_passed = False

# Gate 3: Weight Gate
gate3_passed = True
if not (0.1535 <= hours_eff_weight <= 0.1555):
    print(f"[FAIL] Gate 3: Hours weight {hours_eff_weight*100:.3f}% out of range [15.35%, 15.55%]")
    gate3_passed = False

h_comps = RAY_DALIO_GAUGE_TREES["health_index"]["components"]
h_sum = sum(c["weight"] for c in h_comps)
if not (len(h_comps) == 9 and np.isclose(h_sum, 1.0)):
    print(f"[FAIL] Gate 3: health_index components count={len(h_comps)} (expected 9), sum={h_sum:.4f} (expected 1.0)")
    gate3_passed = False

hap_comps = RAY_DALIO_GAUGE_TREES["happiness_index"]["components"]
hap_sum = sum(c["weight"] for c in hap_comps)
if not (len(hap_comps) == 5 and np.isclose(hap_sum, 1.0)):
    print(f"[FAIL] Gate 3: happiness_index components count={len(hap_comps)} (expected 5), sum={hap_sum:.4f} (expected 1.0)")
    gate3_passed = False

fg_ch = RAY_DALIO_GAUGE_TREES["future_growth"]["children"]
if not np.isclose(fg_ch["productivity_impact"]["weight"], 0.65):
    print(f"[FAIL] Gate 3: future_growth.productivity_impact weight={fg_ch['productivity_impact']['weight']} (expected 0.65)")
    gate3_passed = False
if not np.isclose(fg_ch["indebtedness_impact"]["weight"], 0.35):
    print(f"[FAIL] Gate 3: future_growth.indebtedness_impact weight={fg_ch['indebtedness_impact']['weight']} (expected 0.35)")
    gate3_passed = False

prod_ch = RAY_DALIO_GAUGE_TREES["productivity_gauge"]["children"]
if not np.isclose(prod_ch["what_you_pay_vs_what_you_get"]["weight"], 0.70):
    print(f"[FAIL] Gate 3: productivity_gauge.what_you_pay_vs_what_you_get weight={prod_ch['what_you_pay_vs_what_you_get']['weight']} (expected 0.70)")
    gate3_passed = False
if not np.isclose(prod_ch["culture"]["weight"], 0.30):
    print(f"[FAIL] Gate 3: productivity_gauge.culture weight={prod_ch['culture']['weight']} (expected 0.30)")
    gate3_passed = False

if gate3_passed:
    print("GATE 3 (Weight Gate): PASSED!")
else:
    print("GATE 3 (Weight Gate): FAILED!")
    gates_passed = False

# Gate 4: Data Sanity Gates
# 4(a) IMF.NGDPDPC vs §14.3
PDF_NGDPDPC_35 = {
    "IND": 2.5, "IDN": 4.8, "ZAF": 6.2, "THA": 7.0, "COL": 7.9,
    "BRA": 10.0, "ARG": 10.4, "CHN": 12.7, "MEX": 14.8, "TUR": 14.8,
    "RUS": 14.9, "CHL": 16.3, "POL": 22.7, "HUN": 23.0, "GRC": 24.3,
    "PRT": 29.1, "JPN": 30.8, "CZE": 32.5, "SAU": 33.1, "ESP": 34.5,
    "KOR": 35.6, "ITA": 38.9, "FRA": 47.0, "GBR": 51.2, "ARE": 54.0,
    "DEU": 55.1, "CAN": 55.3, "BEL": 55.5, "SWE": 57.1, "NLD": 66.6,
    "AUS": 67.6, "USA": 83.1, "NOR": 87.4, "SGP": 91.6, "CHE": 101.1
}
gate4a_passed = True
imf_2024 = df_master[(df_master["indicator_code"] == "IMF.NGDPDPC") & (df_master["year"] == 2024)].drop_duplicates("country_code").set_index("country_code")["value"]
if not imf_2024.empty:
    rel_devs = [abs(float(imf_2024[c])/1000.0 - target_k)/target_k for c, target_k in PDF_NGDPDPC_35.items() if c in imf_2024.index]
    if rel_devs:
        med_dev = float(np.median(rel_devs))
        max_dev = float(np.max(rel_devs))
        print(f"Data sanity 4a: median rel dev = {med_dev*100:.2f}%, max = {max_dev*100:.2f}%")
        if med_dev > 0.10:
            print(f"[FAIL] Gate 4(a): Median rel dev {med_dev*100:.2f}% > 10%")
            gate4a_passed = False
        if max_dev > 0.30:
            print(f"[FAIL] Gate 4(a): Max rel dev {max_dev*100:.2f}% > 30%")
            gate4a_passed = False

        if gate4a_passed:
            print("GATE 4(a) (IMF.NGDPDPC Sanity Gate): PASSED!")
        else:
            print("GATE 4(a) (IMF.NGDPDPC Sanity Gate): FAILED!")
            gates_passed = False
    else:
        print("[WARNING] Gate 4(a): No matching countries found in IMF.NGDPDPC 2024.")
else:
    print("[WARNING] Gate 4(a): IMF.NGDPDPC 2024 dataset is empty.")

# 4(b) DERIVED.WORKING_AGE_GROWTH_FWD10 at 2024 vs §14.2
PDF_WA_GROWTH_FWD10 = {
    "IND": 0.9, "ARE": 0.8, "IDN": 0.7, "SAU": 1.6, "TUR": 0.4,
    "CHN": -0.4, "POL": -0.6, "ZAF": 1.0, "RUS": -0.3, "COL": 0.5,
    "SGP": -0.2, "MEX": 0.8, "CHL": 0.2, "SWE": 0.4, "AUS": 0.6,
    "ARG": 0.8, "THA": -0.7, "HUN": -0.5, "CZE": 0.0, "KOR": -1.2,
    "BRA": 0.1, "USA": 0.2, "GBR": 0.4, "NLD": -0.2, "JPN": -0.8,
    "CAN": 0.3, "PRT": -0.8, "BEL": -0.2, "FRA": -0.1, "NOR": 0.1,
    "ESP": -0.7, "CHE": -0.3, "GRC": -0.9, "DEU": -1.1, "ITA": -1.1
}
gate4b_passed = True
wa_2024 = df_master[(df_master["indicator_code"] == "DERIVED.WORKING_AGE_GROWTH_FWD10") & (df_master["year"] == 2024)].drop_duplicates("country_code").set_index("country_code")["value"]
if not wa_2024.empty:
    diffs = [abs(float(wa_2024[c]) - target_wa) for c, target_wa in PDF_WA_GROWTH_FWD10.items() if c in wa_2024.index]
    if diffs:
        within_pct = (np.array(diffs) <= 0.3).mean() * 100.0
        print(f"Data sanity 4b: {within_pct:.1f}% within 0.3 pp.")
        if within_pct < 90.0:
            print(f"[FAIL] Gate 4(b): Only {within_pct:.1f}% within 0.3 pp, expected >= 90%")
            gate4b_passed = False

        if gate4b_passed:
            print("GATE 4(b) (Working Age Growth Sanity Gate): PASSED!")
        else:
            print("GATE 4(b) (Working Age Growth Sanity Gate): FAILED!")
            gates_passed = False
    else:
        print("[WARNING] Gate 4(b): No matching countries found in DERIVED.WORKING_AGE_GROWTH_FWD10 2024.")
else:
    print("[WARNING] Gate 4(b): DERIVED.WORKING_AGE_GROWTH_FWD10 2024 dataset is empty.")

# --- Printed Checks (§14.1, §14.4) ---
print("\n--- Printed Diagnostics (§14.1 & §14.4) ---")
# Headline Empire Scores Comparison (p. 135)
PDF_EMPIRE_SCORES = {
    "USA": 0.89, "CHN": 0.80, "DEU": 0.38, "JPN": 0.33, "KOR": 0.32, "IND": 0.30,
    "GBR": 0.29, "FRA": 0.27, "RUS": 0.26, "SGP": 0.24, "AUS": 0.23, "TUR": 0.21,
    "CAN": 0.21, "CHE": 0.19, "BRA": 0.18, "NLD": 0.17, "IDN": 0.17, "ITA": 0.17,
    "ESP": 0.17, "SAU": 0.15, "MEX": 0.14, "ZAF": 0.10, "ARG": 0.07
}
strength_scores = df_node_scores[
    (df_node_scores["node_path"] == "strength_determinants") & 
    (df_node_scores["year"] == df_node_scores["year"].max())
].set_index("country_code")["score"]

common_entities = [c for c in DALIO_24 if c in strength_scores.index and c in PDF_EMPIRE_SCORES]
if len(common_entities) >= 10:
    calc_vals = [strength_scores[c] for c in common_entities]
    pdf_vals = [PDF_EMPIRE_SCORES[c] for c in common_entities]
    corr, _ = spearmanr(calc_vals, pdf_vals)
    print(f"Strength Gauge vs PDF Empire Scores Spearman Correlation: {corr:.3f} across {len(common_entities)} entities.")

# Summary Status
if gates_passed and structural_passed:
    print("\nALL HARD GATES PASSED! MVP READY.")
else:
    print("\n[COMPLETED WITH FAILURES] One or more checks/gates failed. Review log above.")